# Traffic Forecasting — Professor Review Notebook

This is an **organized reference to the historical experiment code**, not a verified clean-run notebook. The cells are long and computationally expensive; **do not Run All**. Original source cells are reproduced unchanged, with outputs cleared. See `README_START_HERE.md` and `REPRODUCIBILITY_NOTES.md` for model/data paths, findings, and remaining limitations.

**Research scope:** Short-term traffic **speed** forecasting, not traffic signal optimization.

## METR-LA experimental pipeline (historical; requires full clean rerun for method correction)

Source: original notebook cell 78. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# METR-LA FULL-NETWORK XGBOOST + LSTM + HYBRID PIPELINE
# 207 sensors | 15, 30, 60-minute horizons
# Chronological base/fusion/validation separation
# ============================================================

import os
import gc
import json
import time
import copy
import shutil
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

from xgboost import XGBRegressor
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error
from google.colab import files

warnings.filterwarnings(
    "ignore",
    message="Downcasting object dtype arrays on .fillna",
    category=FutureWarning
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Our 15-minute experiment is already completed.
# Run the remaining two horizons by default.
#
# To retrain EVERYTHING from scratch, change to:
# HORIZONS = [3, 6, 12]

HORIZONS = [6, 12]

HORIZON_NAMES = {
    3: "15min",
    6: "30min",
    12: "60min"
}

SEQUENCE_LENGTH = 12
LSTM_EPOCHS = 5
FUSION_EPOCHS = 25

LSTM_BATCH_SIZE = 2048
FUSION_BATCH_SIZE = 4096
INFERENCE_BATCH_SIZE = 8192

SAVE_DIR = "traffic_models/full_network"
os.makedirs(SAVE_DIR, exist_ok=True)

# Require the existing notebook objects
required_objects = [
    "raw_df",
    "clean_full",
    "valid_full",
    "feature_cols",
    "build_sensor_features",
    "TrafficLSTM",
    "HybridFusion",
    "train_end",
    "train_boundary",
    "val_boundary"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing notebook objects: "
        + ", ".join(missing_objects)
    )

print("Device:", DEVICE)
print("Sensors:", clean_full.shape[1])
print("Horizon steps:", HORIZONS)

# Chronological cutoff within the original training period
BASE_CUTOFF = raw_df.index[
    int(train_end * 0.70)
]

LAG_COLUMNS = [
    f"lag_{i}" for i in range(11, -1, -1)
]

LAG_INDICES = [
    feature_cols.index(name)
    for name in LAG_COLUMNS
]


# ============================================================
# 2. EVALUATION FUNCTION
# ============================================================

def evaluate_model(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    # Same nonzero-target MAPE definition
    # used in our earlier experiments.
    mask = np.abs(y_true) > 1e-6

    mape = (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        ) * 100
    )

    return float(mae), float(rmse), float(mape)


# ============================================================
# 3. PREPARE ONE HORIZON ACROSS 207 SENSORS
# ============================================================

def prepare_full_dataset(horizon):

    X_parts = {"train": [], "val": []}
    y_parts = {"train": [], "val": []}

    base_masks = []
    fusion_masks = []

    # Conservative temporal separation
    fusion_start = BASE_CUTOFF + pd.Timedelta(
        minutes=5 * (SEQUENCE_LENGTH + horizon)
    )

    for sensor_idx, sensor_id in enumerate(
        clean_full.columns
    ):

        # Uses current cleaned dataset.
        # All features are historical/current only.
        sensor_features = build_sensor_features(sensor_id)

        target_time = (
            sensor_features.index
            + pd.Timedelta(minutes=5 * horizon)
        )

        valid_targets = sensor_features[
            f"valid_target_{horizon}"
        ].to_numpy(dtype=bool)

        y = sensor_features[
            f"target_{horizon}"
        ].to_numpy(dtype=np.float32)

        X = sensor_features[
            feature_cols
        ].to_numpy(dtype=np.float32)

        train_mask = (
            (target_time < train_boundary)
            & valid_targets
        )

        val_mask = (
            (target_time >= train_boundary)
            & (target_time < val_boundary)
            & valid_targets
        )

        X_parts["train"].append(X[train_mask])
        y_parts["train"].append(y[train_mask])

        X_parts["val"].append(X[val_mask])
        y_parts["val"].append(y[val_mask])

        # Train base models only on older targets
        base_mask = (
            target_time[train_mask] < BASE_CUTOFF
        )

        # Train fusion on later prediction origins
        fusion_mask = (
            sensor_features.index[train_mask]
            >= fusion_start
        )

        base_masks.append(
            np.asarray(base_mask, dtype=bool)
        )

        fusion_masks.append(
            np.asarray(fusion_mask, dtype=bool)
        )

        if (sensor_idx + 1) % 50 == 0:
            print(
                f"Prepared {sensor_idx + 1}/207 sensors",
                flush=True
            )

    X_train = np.concatenate(
        X_parts["train"], axis=0
    )

    y_train = np.concatenate(
        y_parts["train"], axis=0
    )

    X_val = np.concatenate(
        X_parts["val"], axis=0
    )

    y_val = np.concatenate(
        y_parts["val"], axis=0
    )

    base_mask = np.concatenate(base_masks)
    fusion_mask = np.concatenate(fusion_masks)

    assert len(base_mask) == len(X_train)
    assert len(fusion_mask) == len(X_train)
    assert not np.any(base_mask & fusion_mask)

    assert np.isfinite(X_train).all()
    assert np.isfinite(X_val).all()
    assert np.isfinite(y_train).all()
    assert np.isfinite(y_val).all()

    return (
        X_train, y_train,
        X_val, y_val,
        base_mask, fusion_mask,
        fusion_start
    )


# ============================================================
# 4. TRAIN SHARED BASE LSTM
# ============================================================

def train_base_lstm(
    X_base, y_base,
    input_mean, input_std,
    target_mean, target_std
):

    # Extract 12 historical speed values
    X_seq = X_base[:, LAG_INDICES].copy()

    X_seq -= input_mean
    X_seq /= input_std

    X_seq = X_seq.reshape(-1, 12, 1)

    y_seq = (
        (y_base - target_mean) / target_std
    ).astype(np.float32)

    dataset = TensorDataset(
        torch.from_numpy(X_seq),
        torch.from_numpy(y_seq)
    )

    loader = DataLoader(
        dataset,
        batch_size=LSTM_BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    model = TrafficLSTM(
        input_size=1,
        hidden_size=64
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.MSELoss()

    start = time.time()

    for epoch in range(LSTM_EPOCHS):

        model.train()
        total_loss = 0.0

        for xb, yb in loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            pred = model(xb)
            loss = criterion(pred, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)

        avg_loss = total_loss / len(dataset)

        print(
            f"LSTM Epoch {epoch+1:02d}/"
            f"{LSTM_EPOCHS} | "
            f"Loss: {avg_loss:.4f}",
            flush=True
        )

    print(
        "LSTM training time:",
        round(time.time() - start, 2),
        "seconds"
    )

    # Freeze base feature extractor
    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    del dataset, loader, X_seq, y_seq
    gc.collect()

    return model


# ============================================================
# 5. GENERATE FUSION INPUTS IN BATCHES
# ============================================================

def generate_fusion_inputs(
    X_source,
    xgb_predictions,
    lstm_model,
    input_mean,
    input_std,
    target_mean,
    target_std,
    indices=None
):

    total = len(xgb_predictions)

    fusion_inputs = np.empty(
        (total, 65),
        dtype=np.float32
    )

    lstm_model.eval()

    with torch.inference_mode():

        for start in range(
            0, total, INFERENCE_BATCH_SIZE
        ):

            end = min(
                start + INFERENCE_BATCH_SIZE,
                total
            )

            if indices is None:
                batch = X_source[start:end]
            else:
                batch = X_source[
                    indices[start:end]
                ]

            sequences = batch[
                :, LAG_INDICES
            ].copy()

            sequences -= input_mean
            sequences /= input_std

            sequences = sequences.reshape(
                -1, 12, 1
            )

            xb = torch.from_numpy(
                sequences
            ).to(DEVICE)

            lstm_out, _ = lstm_model.lstm(xb)

            hidden = lstm_out[:, -1, :]

            fusion_inputs[start:end, :64] = (
                hidden.cpu().numpy()
            )

            fusion_inputs[start:end, 64] = (
                (xgb_predictions[start:end] - target_mean)
                / target_std
            )

    return fusion_inputs


# ============================================================
# 6. TRAIN FUSION NEURAL NETWORK
# ============================================================

def train_fusion(
    X_train_fusion,
    y_train_fusion,
    X_val_fusion,
    y_val_fusion
):

    train_dataset = TensorDataset(
        torch.from_numpy(X_train_fusion),
        torch.from_numpy(y_train_fusion)
    )

    val_dataset = TensorDataset(
        torch.from_numpy(X_val_fusion),
        torch.from_numpy(y_val_fusion)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=FUSION_BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=INFERENCE_BATCH_SIZE,
        shuffle=False
    )

    model = HybridFusion().to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    wait = 0
    patience = 5

    start = time.time()

    for epoch in range(FUSION_EPOCHS):

        model.train()
        total_train = 0.0

        for xb, yb in train_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            pred = model(xb)
            loss = criterion(pred, yb)

            loss.backward()
            optimizer.step()

            total_train += loss.item() * len(xb)

        train_loss = (
            total_train / len(train_dataset)
        )

        model.eval()
        total_val = 0.0

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                pred = model(xb)
                loss = criterion(pred, yb)

                total_val += loss.item() * len(xb)

        val_loss = total_val / len(val_dataset)

        print(
            f"Fusion Epoch {epoch+1:02d} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f}",
            flush=True
        )

        if val_loss < best_loss - 1e-5:

            best_loss = val_loss
            best_state = copy.deepcopy(
                model.state_dict()
            )

            best_epoch = epoch + 1
            wait = 0

        else:
            wait += 1

            if wait >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_state)
    model.eval()

    print(
        "Fusion training time:",
        round(time.time() - start, 2),
        "seconds"
    )

    # Batched predictions
    prediction_parts = []

    with torch.no_grad():

        for xb, _ in val_loader:

            pred = model(
                xb.to(DEVICE)
            ).cpu().numpy()

            prediction_parts.append(pred)

    predictions = np.concatenate(
        prediction_parts
    )

    return model, predictions, best_epoch


# ============================================================
# 7. COMPLETE MULTI-HORIZON TRAINING
# ============================================================

all_results = []

for horizon in HORIZONS:

    horizon_name = HORIZON_NAMES[horizon]

    print("\n" + "=" * 60)
    print(
        f"STARTING {horizon_name.upper()} "
        "FULL-NETWORK FORECAST"
    )
    print("=" * 60, flush=True)

    start_horizon = time.time()

    # ------------------------------------
    # A. Prepare dataset
    # ------------------------------------

    (
        X_train, y_train,
        X_val, y_val,
        base_mask, fusion_mask,
        fusion_start_time_h
    ) = prepare_full_dataset(horizon)

    base_indices = np.flatnonzero(base_mask)
    fusion_indices = np.flatnonzero(fusion_mask)

    print("Training samples:", len(X_train))
    print("Validation samples:", len(X_val))
    print("Base samples:", len(base_indices))
    print("Fusion samples:", len(fusion_indices))

    # ------------------------------------
    # B. Base-period data
    # ------------------------------------

    X_base = X_train[base_indices]
    y_base = y_train[base_indices]

    # ------------------------------------
    # C. Base XGBoost
    # ------------------------------------

    xgb_model = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=2
    )

    print("\nTraining base XGBoost...", flush=True)

    xgb_model.fit(X_base, y_base)

    # Predict fusion period in chunks
    fusion_xgb_preds = np.empty(
        len(fusion_indices),
        dtype=np.float32
    )

    for start in range(
        0, len(fusion_indices), 100000
    ):

        end = min(
            start + 100000,
            len(fusion_indices)
        )

        fusion_xgb_preds[start:end] = (
            xgb_model.predict(
                X_train[fusion_indices[start:end]]
            )
        )

    val_xgb_preds = xgb_model.predict(X_val)

    # ------------------------------------
    # D. Base normalization
    # ------------------------------------

    base_lags = X_base[:, LAG_INDICES]

    input_mean = float(base_lags.mean())
    input_std = float(base_lags.std()) + 1e-6

    target_mean = float(y_base.mean())
    target_std = float(y_base.std()) + 1e-6

    # ------------------------------------
    # E. Train base LSTM
    # ------------------------------------

    print("\nTraining base LSTM...", flush=True)

    lstm_model = train_base_lstm(
        X_base,
        y_base,
        input_mean,
        input_std,
        target_mean,
        target_std
    )

    # ------------------------------------
    # F. Create fusion inputs
    # ------------------------------------

    print(
        "\nGenerating fusion-training features...",
        flush=True
    )

    X_fusion = generate_fusion_inputs(
        X_train,
        fusion_xgb_preds,
        lstm_model,
        input_mean,
        input_std,
        target_mean,
        target_std,
        indices=fusion_indices
    )

    print(
        "Generating fusion-validation features...",
        flush=True
    )

    X_fusion_val = generate_fusion_inputs(
        X_val,
        val_xgb_preds,
        lstm_model,
        input_mean,
        input_std,
        target_mean,
        target_std
    )

    y_fusion = (
        (y_train[fusion_indices] - target_mean)
        / target_std
    ).astype(np.float32)

    y_fusion_val = (
        (y_val - target_mean)
        / target_std
    ).astype(np.float32)

    print("Fusion train:", X_fusion.shape)
    print("Fusion val:", X_fusion_val.shape)

    # ------------------------------------
    # G. Train fusion network
    # ------------------------------------

    print("\nTraining fusion model...", flush=True)

    (
        fusion_model,
        hybrid_normalized_preds,
        best_epoch
    ) = train_fusion(
        X_fusion,
        y_fusion,
        X_fusion_val,
        y_fusion_val
    )

    hybrid_preds = (
        hybrid_normalized_preds * target_std
        + target_mean
    )

    # ------------------------------------
    # H. Validation metrics
    # ------------------------------------

    baseline_preds = X_val[
        :, feature_cols.index("lag_0")
    ]

    base_metrics = evaluate_model(
        y_val, baseline_preds
    )

    xgb_metrics = evaluate_model(
        y_val, val_xgb_preds
    )

    hybrid_metrics = evaluate_model(
        y_val, hybrid_preds
    )

    improvement = (
        (xgb_metrics[0] - hybrid_metrics[0])
        / xgb_metrics[0]
    ) * 100

    print("\n" + "-" * 60)
    print(f"VALIDATION RESULTS — {horizon_name}")
    print("-" * 60)

    comparison = pd.DataFrame({
        "Model": [
            "Persistence",
            "Base XGBoost",
            "Hybrid XGBoost + LSTM"
        ],
        "MAE": [
            base_metrics[0],
            xgb_metrics[0],
            hybrid_metrics[0]
        ],
        "RMSE": [
            base_metrics[1],
            xgb_metrics[1],
            hybrid_metrics[1]
        ],
        "MAPE (%)": [
            base_metrics[2],
            xgb_metrics[2],
            hybrid_metrics[2]
        ]
    })

    display(comparison.round(4))

    print(
        "Hybrid MAE improvement:",
        f"{improvement:.2f}%"
    )

    # ------------------------------------
    # I. Save models
    # ------------------------------------

    joblib.dump(
        xgb_model,
        f"{SAVE_DIR}/hybrid_base_xgboost_{horizon_name}.pkl"
    )

    torch.save(
        lstm_model.state_dict(),
        f"{SAVE_DIR}/hybrid_base_lstm_{horizon_name}.pth"
    )

    torch.save(
        fusion_model.state_dict(),
        f"{SAVE_DIR}/hybrid_fusion_{horizon_name}.pth"
    )

    result_record = {
        "dataset": "METR-LA",
        "num_sensors": 207,
        "horizon_minutes": int(horizon * 5),
        "evaluation": "validation",
        "validation_samples": int(len(y_val)),
        "persistence_mae": base_metrics[0],
        "persistence_rmse": base_metrics[1],
        "persistence_mape": base_metrics[2],
        "xgboost_mae": xgb_metrics[0],
        "xgboost_rmse": xgb_metrics[1],
        "xgboost_mape": xgb_metrics[2],
        "hybrid_mae": hybrid_metrics[0],
        "hybrid_rmse": hybrid_metrics[1],
        "hybrid_mape": hybrid_metrics[2],
        "improvement_percent": float(improvement),
        "best_fusion_epoch": int(best_epoch),
        "input_mean": float(input_mean),
        "input_std": float(input_std),
        "target_mean": float(target_mean),
        "target_std": float(target_std),
        "feature_names": list(feature_cols),
        "base_cutoff": str(BASE_CUTOFF),
        "fusion_start": str(fusion_start_time_h),
        "sequence_length": 12,
        "lstm_hidden_size": 64
    }

    with open(
        f"{SAVE_DIR}/hybrid_results_{horizon_name}.json",
        "w"
    ) as f:
        json.dump(result_record, f, indent=4)

    all_results.append(result_record)

    print(
        f"\n{horizon_name} models saved successfully!"
    )

    print(
        "Total horizon time:",
        round((time.time() - start_horizon) / 60, 2),
        "minutes"
    )

    # ------------------------------------
    # J. Free large arrays before next horizon
    # ------------------------------------

    del (
        X_train, y_train,
        X_val, y_val,
        X_base, y_base,
        X_fusion, X_fusion_val,
        y_fusion, y_fusion_val,
        base_mask, fusion_mask,
        base_indices, fusion_indices,
        fusion_xgb_preds, val_xgb_preds,
        hybrid_preds, hybrid_normalized_preds,
        xgb_model, lstm_model, fusion_model,
        base_lags, comparison
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 8. SAVE COMBINED RESULT TABLE
# ============================================================

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    f"{SAVE_DIR}/multi_horizon_results.csv",
    index=False
)

print("\n" + "=" * 60)
print("MULTI-HORIZON TRAINING COMPLETED")
print("=" * 60)

display(
    results_df[
        [
            "horizon_minutes",
            "xgboost_mae",
            "hybrid_mae",
            "improvement_percent"
        ]
    ].round(4)
)


# ============================================================
# 9. CREATE AND DOWNLOAD BACKUP
# ============================================================

backup_path = (
    "/content/METR_LA_Multi_Horizon_Models.zip"
)

shutil.make_archive(
    backup_path.replace(".zip", ""),
    "zip",
    root_dir="traffic_models",
    base_dir="full_network"
)

print("\nBACKUP CREATED SUCCESSFULLY!")
print("Backup file:", backup_path)

print("\nAll saved model files:")
for filename in sorted(os.listdir(SAVE_DIR)):
    print(filename)

files.download(backup_path)


## METR-LA test evaluation (verify correct feature order before relying on output)

Source: original notebook cell 90. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# METR-LA FULL-NETWORK TEST EVALUATION
# 207 Sensors | 15, 30, 60-minute predictions
# Persistence vs XGBoost vs LSTM vs Hybrid
# NO RETRAINING
# ============================================================

import os
import json
import gc
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_DIR = "/content/traffic_models/full_network"
RESULT_DIR = "/content/traffic_models/test_results"

os.makedirs(RESULT_DIR, exist_ok=True)

HORIZONS = {
    3: "15min",
    6: "30min",
    12: "60min"
}

LAG_INDICES = [
    feature_cols.index(f"lag_{i}")
    for i in range(11, -1, -1)
]

assert len(feature_cols) == 24
assert clean_full.shape == (34272, 207)

print("Device:", DEVICE)
print("Sensors:", len(clean_full.columns))
print("Original preprocessing verified.")


# ------------------------------------------------------------
# 2. ORIGINAL LSTM ARCHITECTURE
# ------------------------------------------------------------

class TrafficLSTM(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=64
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size, 1
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        return self.fc(
            output[:, -1, :]
        ).squeeze(-1)


# ------------------------------------------------------------
# 3. CORRECTED FUSION ARCHITECTURE
# IMPORTANT: ORIGINAL ATTRIBUTE NAME = network
# ------------------------------------------------------------

class HybridFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(65, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


# ------------------------------------------------------------
# 4. METRICS
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(
        y_true, dtype=np.float32
    )

    y_pred = np.asarray(
        y_pred, dtype=np.float32
    )

    mae = mean_absolute_error(
        y_true, y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true, y_pred
        )
    )

    mask = np.abs(y_true) > 1e-6

    mape = 100 * np.mean(
        np.abs(
            (y_true[mask] - y_pred[mask])
            / y_true[mask]
        )
    )

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "MAPE (%)": float(mape)
    }


# ------------------------------------------------------------
# 5. CREATE TEST DATA FROM ORIGINAL CLEANED DATASET
# ------------------------------------------------------------

def prepare_test_data(horizon):

    X_parts = []
    y_parts = []

    for sensor_id in clean_full.columns:

        data = build_sensor_features(
            sensor_id
        )

        target_time = (
            data.index
            + pd.Timedelta(minutes=5 * horizon)
        )

        valid_target = data[
            f"valid_target_{horizon}"
        ].to_numpy(dtype=bool)

        test_mask = np.asarray(
            (target_time >= val_boundary)
            & valid_target,
            dtype=bool
        )

        X_parts.append(
            data.loc[
                test_mask, feature_cols
            ].to_numpy(dtype=np.float32)
        )

        y_parts.append(
            data.loc[
                test_mask, f"target_{horizon}"
            ].to_numpy(dtype=np.float32)
        )

    X_test = np.concatenate(
        X_parts, axis=0
    )

    y_test = np.concatenate(
        y_parts, axis=0
    )

    assert X_test.shape[1] == 24
    assert len(X_test) == len(y_test)

    assert np.isfinite(X_test).all()
    assert np.isfinite(y_test).all()

    return X_test, y_test


# ------------------------------------------------------------
# 6. LSTM + HYBRID INFERENCE
# ------------------------------------------------------------

def predict_lstm_hybrid(
    X_test,
    xgb_preds,
    lstm_model,
    fusion_model,
    metadata,
    batch_size=8192
):

    input_mean = float(
        metadata["input_mean"]
    )

    input_std = float(
        metadata["input_std"]
    )

    target_mean = float(
        metadata["target_mean"]
    )

    target_std = float(
        metadata["target_std"]
    )

    n = len(X_test)

    lstm_predictions = np.empty(
        n, dtype=np.float32
    )

    hybrid_predictions = np.empty(
        n, dtype=np.float32
    )

    lstm_model.eval()
    fusion_model.eval()

    with torch.inference_mode():

        for start in range(
            0, n, batch_size
        ):

            end = min(
                start + batch_size, n
            )

            batch = X_test[start:end]

            # Original LSTM input:
            # oldest lag -> newest lag
            sequences = batch[
                :, LAG_INDICES
            ].copy()

            sequences -= input_mean
            sequences /= input_std

            sequences = sequences.reshape(
                -1, 12, 1
            )

            tensor = torch.from_numpy(
                sequences
            ).to(DEVICE)

            # Last-step LSTM representation
            lstm_output, _ = (
                lstm_model.lstm(tensor)
            )

            hidden = lstm_output[
                :, -1, :
            ]

            # Standalone LSTM prediction
            lstm_norm = (
                lstm_model.fc(hidden)
                .squeeze(-1)
            )

            # XGBoost input to fusion model
            xgb_normalized = (
                xgb_preds[start:end] - target_mean
            ) / target_std

            xgb_tensor = torch.from_numpy(
                xgb_normalized.astype(np.float32)
            ).to(DEVICE).unsqueeze(1)

            # 64 LSTM features + 1 XGBoost prediction
            fusion_input = torch.cat(
                [hidden, xgb_tensor],
                dim=1
            )

            fusion_norm = fusion_model(
                fusion_input
            )

            # Reverse target normalization
            lstm_predictions[start:end] = (
                lstm_norm.cpu().numpy()
                * target_std + target_mean
            )

            hybrid_predictions[start:end] = (
                fusion_norm.cpu().numpy()
                * target_std + target_mean
            )

    return lstm_predictions, hybrid_predictions


# ------------------------------------------------------------
# 7. RUN TEST EVALUATION FOR ALL HORIZONS
# ------------------------------------------------------------

all_results = []

for horizon, name in HORIZONS.items():

    print("\n" + "=" * 60)
    print(f"TEST EVALUATION — {name}")
    print("=" * 60, flush=True)

    # Load saved metadata
    results_path = os.path.join(
        MODEL_DIR,
        f"hybrid_results_{name}.json"
    )

    with open(results_path, "r") as f:
        metadata = json.load(f)

    # 15-minute metadata was saved separately
    if any(
        key not in metadata
        for key in [
            "input_mean",
            "input_std",
            "target_mean",
            "target_std"
        ]
    ):

        base_metadata_path = os.path.join(
            MODEL_DIR,
            f"hybrid_base_metadata_{name}.json"
        )

        if not os.path.isfile(base_metadata_path):
            raise FileNotFoundError(
                "Missing normalization statistics "
                f"for {name}"
            )

        with open(
            base_metadata_path, "r"
        ) as f:
            base_metadata = json.load(f)

        metadata.update(base_metadata)

    # Load XGBoost model
    xgb_model = joblib.load(
        os.path.join(
            MODEL_DIR,
            f"hybrid_base_xgboost_{name}.pkl"
        )
    )

    # Load saved LSTM weights
    lstm_model = TrafficLSTM().to(DEVICE)

    lstm_state = torch.load(
        os.path.join(
            MODEL_DIR,
            f"hybrid_base_lstm_{name}.pth"
        ),
        map_location=DEVICE,
        weights_only=True
    )

    lstm_model.load_state_dict(
        lstm_state
    )

    # Load saved fusion weights
    fusion_model = HybridFusion().to(DEVICE)

    fusion_state = torch.load(
        os.path.join(
            MODEL_DIR,
            f"hybrid_fusion_{name}.pth"
        ),
        map_location=DEVICE,
        weights_only=True
    )

    fusion_model.load_state_dict(
        fusion_state
    )

    print("All models loaded.")

    # Construct held-out test features
    X_test, y_test = prepare_test_data(
        horizon
    )

    print(
        "Test observations:",
        len(y_test),
        flush=True
    )

    # Persistence predictions
    persistence_preds = X_test[
        :, feature_cols.index("lag_0")
    ]

    # Base XGBoost predictions
    xgb_preds = xgb_model.predict(
        X_test
    ).astype(np.float32)

    # LSTM and hybrid predictions
    (
        lstm_preds,
        hybrid_preds
    ) = predict_lstm_hybrid(
        X_test,
        xgb_preds,
        lstm_model,
        fusion_model,
        metadata
    )

    predictions = {
        "Persistence": persistence_preds,
        "XGBoost": xgb_preds,
        "LSTM": lstm_preds,
        "Hybrid": hybrid_preds
    }

    # Evaluate each model
    for model_name, preds in predictions.items():

        metrics = calculate_metrics(
            y_test, preds
        )

        all_results.append({
            "Horizon": name,
            "Model": model_name,
            "Samples": len(y_test),
            **metrics
        })

        print(
            f"{model_name:12s} | "
            f"MAE: {metrics['MAE']:.4f} | "
            f"RMSE: {metrics['RMSE']:.4f} | "
            f"MAPE: {metrics['MAPE (%)']:.4f}%"
        )

    # Clean up before next horizon
    del (
        X_test, y_test,
        xgb_model,
        lstm_model,
        fusion_model,
        xgb_preds,
        lstm_preds,
        hybrid_preds,
        persistence_preds
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 8. CREATE COMPARISON TABLE
# ------------------------------------------------------------

test_results_df = pd.DataFrame(
    all_results
)

print("\n" + "=" * 60)
print("METR-LA FULL-NETWORK TEST RESULTS")
print("=" * 60)

display(
    test_results_df.round(4)
)

# Save results
csv_path = os.path.join(
    RESULT_DIR,
    "metr_la_test_comparison.csv"
)

test_results_df.to_csv(
    csv_path,
    index=False
)

print("\nResults saved:", csv_path)
print("All three test horizons evaluated successfully!")


# ------------------------------------------------------------
# 9. MAE AND RMSE COMPARISON TABLES
# ------------------------------------------------------------

print("\nMAE COMPARISON")
display(
    test_results_df.pivot(
        index="Horizon",
        columns="Model",
        values="MAE"
    ).round(4)
)

print("\nRMSE COMPARISON")
display(
    test_results_df.pivot(
        index="Horizon",
        columns="Model",
        values="RMSE"
    ).round(4)
)

print("\nMAPE COMPARISON")
display(
    test_results_df.pivot(
        index="Horizon",
        columns="Model",
        values="MAPE (%)"
    ).round(4)
)


## METR-LA XGBoost SHAP explanation

Source: original notebook cell 103. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ==========================================================
# METR-LA SHAP EXPLAINABILITY ANALYSIS
# 207 sensors | 15, 30, 60 minutes
# Explains XGBoost branch, not complete hybrid
# ==========================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

MODEL_DIR = "/content/traffic_models/full_network"
OUTPUT_DIR = "/content/traffic_models/shap_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify original training feature order
with open(
    os.path.join(MODEL_DIR, "metadata_15min.json")
) as f:
    original_features = json.load(f)["features"]

assert list(feature_cols) == list(original_features)

HORIZONS = {
    3: "15min",
    6: "30min",
    12: "60min"
}

# Reproducible sample
rng = np.random.default_rng(42)

all_importances = []

for horizon, name in HORIZONS.items():

    print("\n" + "=" * 50)
    print(f"SHAP ANALYSIS — {name}")
    print("=" * 50)

    model = joblib.load(
        os.path.join(
            MODEL_DIR,
            f"hybrid_base_xgboost_{name}.pkl"
        )
    )

    # Validation observations, not test.
    # Use one sensor-balanced subset for explanations.
    frames = []

    for sensor_id in clean_full.columns:

        data = build_sensor_features(sensor_id)

        target_time = (
            data.index
            + pd.Timedelta(minutes=5 * horizon)
        )

        valid_target = data[
            f"valid_target_{horizon}"
        ].to_numpy(dtype=bool)

        mask = np.asarray(
            (target_time >= train_boundary)
            & (target_time < val_boundary)
            & valid_target
        )

        eligible = data.loc[
            mask, feature_cols
        ]

        # Up to 5 samples per sensor
        if len(eligible) > 0:

            sample_n = min(5, len(eligible))

            sample_indices = rng.choice(
                len(eligible),
                size=sample_n,
                replace=False
            )

            frames.append(
                eligible.iloc[sample_indices]
            )

    X_sample = pd.concat(
        frames,
        ignore_index=True
    ).astype(np.float32)

    print("SHAP samples:", len(X_sample))

    # Tree-based SHAP
    explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(
        X_sample.to_numpy()
    )

    mean_importance = np.mean(
        np.abs(shap_values),
        axis=0
    )

    for feature, importance in zip(
        feature_cols,
        mean_importance
    ):

        all_importances.append({
            "Horizon": name,
            "Feature": feature,
            "Mean absolute SHAP": float(importance)
        })

    ranking = pd.DataFrame({
        "Feature": feature_cols,
        "Mean absolute SHAP": mean_importance
    }).sort_values(
        "Mean absolute SHAP",
        ascending=False
    )

    print("\nTOP 10 FEATURES")
    display(ranking.head(10).round(4))

    # Save each horizon's SHAP values
    ranking.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"shap_importance_{name}.csv"
        ),
        index=False
    )

# ==========================================================
# COMPARISON ACROSS HORIZONS
# ==========================================================

shap_df = pd.DataFrame(all_importances)

comparison = shap_df.pivot(
    index="Feature",
    columns="Horizon",
    values="Mean absolute SHAP"
)

comparison = comparison[
    ["15min", "30min", "60min"]
]

comparison["Overall"] = comparison.mean(axis=1)

comparison = comparison.sort_values(
    "Overall",
    ascending=False
)

print("\n" + "=" * 50)
print("SHAP COMPARISON ACROSS HORIZONS")
print("=" * 50)

display(comparison.head(15).round(4))

# ==========================================================
# PUBLICATION GRAPH
# ==========================================================

top_features = comparison.head(10).index

plot_data = comparison.loc[
    top_features,
    ["15min", "30min", "60min"]
]

ax = plot_data.plot(
    kind="barh",
    figsize=(11, 7)
)

ax.invert_yaxis()

plt.title(
    "METR-LA: SHAP Feature Importance Across Forecast Horizons"
)

plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Input Feature")
plt.legend(title="Forecast Horizon")
plt.tight_layout()

plot_path = os.path.join(
    OUTPUT_DIR,
    "metr_la_shap_horizon_comparison.png"
)

plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save combined results
comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "metr_la_shap_comparison.csv"
    )
)

print("\nSHAP analysis completed!")
print("Saved graph:", plot_path)
print("Saved results in:", OUTPUT_DIR)


## PEMS-BAY DST verification (diagnostic)

Source: original notebook cell 109. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

import pandas as pd
import numpy as np

print("PEMS-BAY DST CORRECTION")
print("=" * 55)

# Preserve original local timestamps
pems_original_index = pems_raw.index.copy()

# Interpret naive clock times as California local time
# March 12, 2017, 2:00-2:59 AM did not exist.
pems_raw.index = pems_raw.index.tz_localize(
    "America/Los_Angeles",
    ambiguous="raise",
    nonexistent="raise"
)

# Verify elapsed-time intervals
elapsed_diffs = pems_raw.index.to_series().diff().dropna()

irregular_elapsed = int(
    (
        elapsed_diffs != pd.Timedelta(minutes=5)
    ).sum()
)

print("Original observations:", len(pems_raw))
print("Timezone:", pems_raw.index.tz)
print("First timestamp:", pems_raw.index.min())
print("Last timestamp:", pems_raw.index.max())

print("\nElapsed-time interval check")
print("-" * 55)
print("Non-5-minute intervals:", irregular_elapsed)

# Inspect DST transition explicitly
dst_window = pems_raw.loc[
    "2017-03-12 01:45":
    "2017-03-12 03:15"
]

print("\nDST transition timestamps:")
for timestamp in dst_window.index:
    print(timestamp)

assert irregular_elapsed == 0, (
    "Additional timestamp irregularities detected."
)

assert len(pems_raw) == 52116

print("\nDST correction verified!")
print("No traffic observations inserted or removed.")


## PEMS-BAY training, all horizons

Source: original notebook cell 111. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# PEMS-BAY COMPLETE XGBOOST + LSTM + HYBRID IMPLEMENTATION
# 325 Sensors | 15, 30, 60-minute forecasting
# DST corrected | Chronological splits | SHAP-ready models
# No Google Drive required
# ============================================================

import os
import gc
import json
import time
import copy
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib

from xgboost import XGBRegressor
from torch.utils.data import TensorDataset, DataLoader
from google.colab import files

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Use all horizons.
# For example, set [3] to test only 15 minutes.
HORIZONS = [3, 6, 12]

HORIZON_NAMES = {
    3: "15min",
    6: "30min",
    12: "60min"
}

DATA_PATH = "/content/pems-bay-dst-corrected.h5"

SAVE_DIR = "/content/traffic_models/pems_bay"
os.makedirs(SAVE_DIR, exist_ok=True)

SEQ_LEN = 12
HIDDEN_SIZE = 64

LSTM_EPOCHS = 5
FUSION_EPOCHS = 25

LSTM_BATCH = 2048
FUSION_BATCH = 4096
PRED_BATCH = 8192

ZERO_AS_MISSING = True

print("=" * 65)
print("PEMS-BAY TRAFFIC FORECASTING")
print("=" * 65)

print("Device:", DEVICE)
print("Horizons:", [h * 5 for h in HORIZONS])
print("Zero-as-missing:", ZERO_AS_MISSING)


# ============================================================
# 2. LOAD DATASET AND CORRECT DST IF NECESSARY
# ============================================================

if not os.path.exists(DATA_PATH):

    original_path = "/content/pems-bay.h5"

    if not os.path.exists(original_path):
        raise FileNotFoundError(
            "Neither PEMS-BAY file exists. "
            "Download pems-bay.h5 first."
        )

    DATA_PATH = original_path

pems_raw = pd.read_hdf(DATA_PATH).astype(
    np.float32
)

# PEMS-BAY timestamps represent California local time.
if pems_raw.index.tz is None:

    pems_raw.index = pems_raw.index.tz_localize(
        "America/Los_Angeles",
        ambiguous="raise",
        nonexistent="raise"
    )

assert pems_raw.shape[1] == 325
assert pems_raw.index.is_unique
assert pems_raw.index.is_monotonic_increasing

intervals = pems_raw.index.to_series().diff().dropna()

assert (
    intervals == pd.Timedelta(minutes=5)
).all(), "Unexpected elapsed-time gap."

print("\nDATASET INFORMATION")
print("-" * 55)

print("Shape:", pems_raw.shape)
print("First timestamp:", pems_raw.index[0])
print("Last timestamp:", pems_raw.index[-1])
print("Timezone:", pems_raw.index.tz)
print("Irregular elapsed intervals:", 0)


# ============================================================
# 3. CHRONOLOGICAL SPLITS
# ============================================================

n_times = len(pems_raw)

TRAIN_END = int(n_times * 0.70)
VAL_END = int(n_times * 0.80)
BASE_END = int(TRAIN_END * 0.70)

BASE_CUTOFF = pems_raw.index[BASE_END]
TRAIN_CUTOFF = pems_raw.index[TRAIN_END]
VAL_CUTOFF = pems_raw.index[VAL_END]

assert BASE_CUTOFF < TRAIN_CUTOFF < VAL_CUTOFF

print("\nCHRONOLOGICAL PARTITIONS")
print("-" * 55)

print("Base model cutoff:", BASE_CUTOFF)
print("Training cutoff:", TRAIN_CUTOFF)
print("Validation cutoff:", VAL_CUTOFF)


# ============================================================
# 4. PREPROCESSING
# ============================================================

raw_values = pems_raw.to_numpy(
    dtype=np.float32,
    copy=False
)

valid_values = np.isfinite(raw_values)

if ZERO_AS_MISSING:
    valid_values = (
        valid_values & (raw_values != 0)
    )

zero_count = int((raw_values == 0).sum())

print("\nDATA QUALITY")
print("-" * 55)

print("Zero readings:", zero_count)
print(
    "Zero percentage:",
    round(
        100 * zero_count / raw_values.size,
        5
    ),
    "%"
)

print(
    "NaN readings:",
    int(np.isnan(raw_values).sum())
)

# Only base-training observations are used
# to calculate imputation medians.
missing_df = pems_raw.mask(~valid_values)

base_missing = missing_df.iloc[:BASE_END]

sensor_medians = base_missing.median()

global_median = float(
    base_missing.stack().median()
)

fallback_values = sensor_medians.fillna(
    global_median
)

# Causal forward-fill, up to six observations.
# Forward filling uses past values only.
clean_df = (
    missing_df
    .ffill(limit=6)
    .fillna(fallback_values)
    .astype(np.float32)
)

assert np.isfinite(
    clean_df.to_numpy()
).all()

clean_values = clean_df.to_numpy(
    dtype=np.float32,
    copy=False
)

timestamps = clean_df.index

print("\nPREPROCESSING VERIFIED")
print("-" * 55)

print("Cleaned dataset:", clean_df.shape)
print("Missing values remaining:", int(
    clean_df.isna().sum().sum()
))
print("Fallback statistics: Base period only")


# ============================================================
# 5. FIXED FEATURE ORDER
# ============================================================

# IMPORTANT:
# Same feature ordering as original METR-LA models.
# Rolling means and standard deviations alternate.

FEATURE_COLS = [
    f"lag_{i}" for i in range(12)
] + [
    "rolling_mean_3",
    "rolling_std_3",
    "rolling_mean_6",
    "rolling_std_6",
    "rolling_mean_12",
    "rolling_std_12",
    "trend_3",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "weekend"
]

assert len(FEATURE_COLS) == 24

LAG_INDICES = [
    FEATURE_COLS.index(f"lag_{i}")
    for i in range(11, -1, -1)
]

print("\nFeature count:", len(FEATURE_COLS))
print("Feature ordering verified.")


# ============================================================
# 6. CALENDAR FEATURES
# ============================================================

# Calendar information uses California local clock time.
hours = (
    timestamps.hour.to_numpy()
    + timestamps.minute.to_numpy() / 60.0
)

days = timestamps.dayofweek.to_numpy()

calendar = {
    "hour_sin": np.sin(2 * np.pi * hours / 24),
    "hour_cos": np.cos(2 * np.pi * hours / 24),
    "day_sin": np.sin(2 * np.pi * days / 7),
    "day_cos": np.cos(2 * np.pi * days / 7),
    "weekend": (days >= 5).astype(np.float32)
}


# ============================================================
# 7. SENSOR FEATURE GENERATOR
# ============================================================

def build_features(sensor_index):

    speed = pd.Series(
        clean_values[:, sensor_index],
        index=timestamps
    )

    data = pd.DataFrame(index=timestamps)

    for lag in range(12):
        data[f"lag_{lag}"] = speed.shift(lag)

    for window in [3, 6, 12]:

        roll = speed.rolling(window)

        data[f"rolling_mean_{window}"] = roll.mean()
        data[f"rolling_std_{window}"] = roll.std()

    data["trend_3"] = speed - speed.shift(3)

    for name, values in calendar.items():
        data[name] = values

    return data[FEATURE_COLS]


# ============================================================
# 8. PREPARE ONE HORIZON
# ============================================================

def prepare_horizon(horizon):

    # Split by actual target timestamp,
    # not by sensor-major array row position.

    fusion_start = (
        BASE_CUTOFF
        + pd.Timedelta(
            minutes=5 * (SEQ_LEN + horizon)
        )
    )

    arrays = {
        name: {"X": [], "y": []}
        for name in ["base", "fusion", "val", "test"]
    }

    target_times = timestamps + pd.Timedelta(
        minutes=5 * horizon
    )

    for sensor_index in range(325):

        features = build_features(sensor_index)

        X = features.to_numpy(
            dtype=np.float32
        )

        speeds = pd.Series(
            clean_values[:, sensor_index],
            index=timestamps
        )

        y = speeds.shift(
            -horizon
        ).to_numpy(dtype=np.float32)

        # Only originally observed future targets.
        valid_target = np.zeros(
            n_times,
            dtype=bool
        )

        valid_target[:-horizon] = (
            valid_values[
                horizon:,
                sensor_index
            ]
        )

        usable = np.zeros(
            n_times,
            dtype=bool
        )

        usable[11:n_times - horizon] = True
        usable &= valid_target

        masks = {
            "base": (
                usable
                & (target_times < BASE_CUTOFF)
            ),

            "fusion": (
                usable
                & (timestamps >= fusion_start)
                & (target_times < TRAIN_CUTOFF)
            ),

            "val": (
                usable
                & (target_times >= TRAIN_CUTOFF)
                & (target_times < VAL_CUTOFF)
            ),

            "test": (
                usable
                & (target_times >= VAL_CUTOFF)
            )
        }

        for split, mask in masks.items():

            arrays[split]["X"].append(
                X[mask]
            )

            arrays[split]["y"].append(
                y[mask]
            )

        if (sensor_index + 1) % 65 == 0:
            print(
                f"Prepared {sensor_index + 1}/325 sensors",
                flush=True
            )

        del features, X, y

    output = {}

    for split in arrays:

        X = np.concatenate(
            arrays[split]["X"],
            axis=0
        )

        y = np.concatenate(
            arrays[split]["y"],
            axis=0
        )

        assert len(X) == len(y)
        assert X.shape[1] == 24
        assert np.isfinite(X).all()
        assert np.isfinite(y).all()

        output[split] = (X, y)

        print(
            split.upper(),
            "samples:",
            len(y)
        )

    output["fusion_start"] = str(
        fusion_start
    )

    return output


# ============================================================
# 9. LSTM MODEL
# ============================================================

class TrafficLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x):

        output, _ = self.lstm(x)

        return self.fc(
            output[:, -1, :]
        ).squeeze(-1)


# ============================================================
# 10. HYBRID FUSION MODEL
# ============================================================

class HybridFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(65, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        return self.network(x).squeeze(-1)


# ============================================================
# 11. METRICS
# ============================================================

def metrics(y, predictions):

    error = (
        y.astype(np.float32)
        - predictions.astype(np.float32)
    )

    mae = float(
        np.mean(
            np.abs(error),
            dtype=np.float64
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                np.square(error),
                dtype=np.float64
            )
        )
    )

    mask = np.abs(y) > 1e-6

    mape = float(
        100 * np.mean(
            np.abs(error[mask] / y[mask]),
            dtype=np.float64
        )
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    }


# ============================================================
# 12. TRAIN BASE LSTM
# ============================================================

def train_base_lstm(X_base, y_base, stats):

    X_seq = X_base[
        :, LAG_INDICES
    ].copy()

    X_seq -= stats["input_mean"]
    X_seq /= stats["input_std"]

    X_seq = X_seq.reshape(-1, 12, 1)

    y_seq = (
        (y_base - stats["target_mean"])
        / stats["target_std"]
    ).astype(np.float32)

    dataset = TensorDataset(
        torch.from_numpy(X_seq),
        torch.from_numpy(y_seq)
    )

    loader = DataLoader(
        dataset,
        batch_size=LSTM_BATCH,
        shuffle=True,
        num_workers=0
    )

    model = TrafficLSTM().to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    loss_fn = nn.MSELoss()

    start_time = time.time()

    for epoch in range(LSTM_EPOCHS):

        model.train()
        total_loss = 0.0

        for xb, yb in loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            prediction = model(xb)

            loss = loss_fn(
                prediction,
                yb
            )

            loss.backward()
            optimizer.step()

            total_loss += (
                loss.item() * len(xb)
            )

        epoch_loss = (
            total_loss / len(dataset)
        )

        print(
            f"LSTM Epoch {epoch+1}/{LSTM_EPOCHS}",
            f"Loss: {epoch_loss:.5f}",
            flush=True
        )

    print(
        "LSTM training seconds:",
        round(
            time.time() - start_time, 2
        )
    )

    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    del loader, dataset, X_seq, y_seq
    gc.collect()

    return model


# ============================================================
# 13. BATCHED XGBOOST
# ============================================================

def xgb_predict(model, X):

    output = np.empty(
        len(X),
        dtype=np.float32
    )

    for start in range(
        0, len(X), 100000
    ):

        end = min(
            start + 100000,
            len(X)
        )

        output[start:end] = model.predict(
            X[start:end]
        )

    return output


# ============================================================
# 14. FUSION FEATURE EXTRACTION
# ============================================================

def fusion_features(
    X, xgb_preds, lstm, stats
):

    output_features = np.empty(
        (len(X), 65),
        dtype=np.float32
    )

    lstm.eval()

    with torch.inference_mode():

        for start in range(
            0, len(X), PRED_BATCH
        ):

            end = min(
                start + PRED_BATCH,
                len(X)
            )

            seq = X[
                start:end, LAG_INDICES
            ].copy()

            seq -= stats["input_mean"]
            seq /= stats["input_std"]

            tensor = torch.from_numpy(
                seq.reshape(-1, 12, 1)
            ).to(DEVICE)

            lstm_output, _ = lstm.lstm(
                tensor
            )

            output_features[start:end, :64] = (
                lstm_output[:, -1, :]
                .cpu()
                .numpy()
            )

            output_features[start:end, 64] = (
                (
                    xgb_preds[start:end]
                    - stats["target_mean"]
                ) / stats["target_std"]
            )

    return output_features


# ============================================================
# 15. TRAIN FUSION
# ============================================================

def train_fusion(
    X_train, y_train,
    X_val, y_val
):

    train_set = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train)
    )

    val_set = TensorDataset(
        torch.from_numpy(X_val),
        torch.from_numpy(y_val)
    )

    train_loader = DataLoader(
        train_set,
        batch_size=FUSION_BATCH,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_set,
        batch_size=PRED_BATCH,
        shuffle=False,
        num_workers=0
    )

    model = HybridFusion().to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_state = None
    best_epoch = 0

    patience = 5
    wait = 0

    start_time = time.time()

    for epoch in range(FUSION_EPOCHS):

        model.train()
        train_total = 0.0

        for xb, yb in train_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            prediction = model(xb)
            loss = criterion(prediction, yb)

            loss.backward()
            optimizer.step()

            train_total += (
                loss.item() * len(xb)
            )

        model.eval()
        val_total = 0.0

        with torch.inference_mode():

            for xb, yb in val_loader:

                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                pred = model(xb)
                loss = criterion(pred, yb)

                val_total += (
                    loss.item() * len(xb)
                )

        train_loss = train_total / len(train_set)
        val_loss = val_total / len(val_set)

        print(
            f"Fusion Epoch {epoch+1:02d}",
            f"Train: {train_loss:.5f}",
            f"Val: {val_loss:.5f}",
            flush=True
        )

        if val_loss < best_loss - 1e-5:

            best_loss = val_loss
            best_state = copy.deepcopy(
                model.state_dict()
            )

            best_epoch = epoch + 1
            wait = 0

        else:
            wait += 1

            if wait >= patience:
                print(
                    "Early stopping triggered.",
                    flush=True
                )
                break

    model.load_state_dict(best_state)
    model.eval()

    print(
        "Fusion training seconds:",
        round(time.time() - start_time, 2)
    )

    return model, best_epoch


# ============================================================
# 16. LSTM + HYBRID INFERENCE
# ============================================================

def predict_lstm_hybrid(
    X, xgb_predictions, lstm, fusion, stats
):

    n = len(X)

    lstm_preds = np.empty(
        n, dtype=np.float32
    )

    hybrid_preds = np.empty(
        n, dtype=np.float32
    )

    lstm.eval()
    fusion.eval()

    with torch.inference_mode():

        for start in range(
            0, n, PRED_BATCH
        ):

            end = min(
                start + PRED_BATCH,
                n
            )

            seq = X[
                start:end, LAG_INDICES
            ].copy()

            seq -= stats["input_mean"]
            seq /= stats["input_std"]

            tensor = torch.from_numpy(
                seq.reshape(-1, 12, 1)
            ).to(DEVICE)

            output, _ = lstm.lstm(tensor)

            hidden = output[:, -1, :]

            lstm_norm = lstm.fc(
                hidden
            ).squeeze(-1)

            xgb_norm = (
                (
                    xgb_predictions[start:end]
                    - stats["target_mean"]
                ) / stats["target_std"]
            ).astype(np.float32)

            xgb_tensor = torch.from_numpy(
                xgb_norm
            ).to(DEVICE).unsqueeze(1)

            combined = torch.cat(
                [hidden, xgb_tensor],
                dim=1
            )

            hybrid_norm = fusion(combined)

            lstm_preds[start:end] = (
                lstm_norm.cpu().numpy()
                * stats["target_std"]
                + stats["target_mean"]
            )

            hybrid_preds[start:end] = (
                hybrid_norm.cpu().numpy()
                * stats["target_std"]
                + stats["target_mean"]
            )

    return lstm_preds, hybrid_preds


# ============================================================
# 17. SAVE JSON
# ============================================================

def save_json(path, data):

    with open(path, "w") as f:
        json.dump(data, f, indent=4)


# ============================================================
# 18. TRAIN AND EVALUATE EACH HORIZON
# ============================================================

all_results = []

for horizon in HORIZONS:

    name = HORIZON_NAMES[horizon]

    print("\n" + "=" * 65)
    print(f"STARTING {name} PEMS-BAY EXPERIMENT")
    print("=" * 65, flush=True)

    xgb_path = os.path.join(
        SAVE_DIR,
        f"xgboost_{name}.pkl"
    )

    lstm_path = os.path.join(
        SAVE_DIR,
        f"lstm_{name}.pth"
    )

    fusion_path = os.path.join(
        SAVE_DIR,
        f"hybrid_{name}.pth"
    )

    metadata_path = os.path.join(
        SAVE_DIR,
        f"metadata_{name}.json"
    )

    result_path = os.path.join(
        SAVE_DIR,
        f"results_{name}.json"
    )

    expected_paths = [
        xgb_path,
        lstm_path,
        fusion_path,
        metadata_path,
        result_path
    ]

    # Skip finished horizons only when all files exist.
    if all(os.path.isfile(p) for p in expected_paths):

        print("Existing complete horizon found. Skipping.")

        with open(result_path) as f:
            stored = json.load(f)

        all_results.extend(
            stored["metrics"]
        )
        continue

    start_horizon = time.time()

    # --------------------------------------------------------
    # A. Build horizon-specific data
    # --------------------------------------------------------

    data = prepare_horizon(horizon)

    X_base, y_base = data["base"]
    X_fusion_raw, y_fusion_raw = data["fusion"]
    X_val, y_val = data["val"]
    X_test, y_test = data["test"]

    # --------------------------------------------------------
    # B. Base-only normalization
    # --------------------------------------------------------

    base_lags = X_base[
        :, LAG_INDICES
    ]

    stats = {
        "input_mean": float(base_lags.mean()),
        "input_std": float(base_lags.std() + 1e-6),
        "target_mean": float(y_base.mean()),
        "target_std": float(y_base.std() + 1e-6)
    }

    del base_lags

    # --------------------------------------------------------
    # C. XGBoost
    # --------------------------------------------------------

    print("\nTraining XGBoost...", flush=True)

    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        objective="reg:squarederror",
        eval_metric="mae",
        random_state=SEED,
        n_jobs=2
    )

    if os.path.isfile(xgb_path):

        print("Loading saved XGBoost.")
        xgb = joblib.load(xgb_path)

    else:

        xgb.fit(X_base, y_base)

        joblib.dump(xgb, xgb_path)
        print("XGBoost saved.")

    # --------------------------------------------------------
    # D. LSTM
    # --------------------------------------------------------

    print("\nTraining/loading LSTM...", flush=True)

    if os.path.isfile(lstm_path):

        lstm = TrafficLSTM().to(DEVICE)

        lstm.load_state_dict(
            torch.load(
                lstm_path,
                map_location=DEVICE,
                weights_only=True
            )
        )

        lstm.eval()

        for p in lstm.parameters():
            p.requires_grad_(False)

        print("Saved LSTM loaded.")

    else:

        lstm = train_base_lstm(
            X_base,
            y_base,
            stats
        )

        torch.save(
            lstm.state_dict(),
            lstm_path
        )

        print("LSTM saved.")

    # Free base arrays
    del X_base, y_base
    gc.collect()

    # --------------------------------------------------------
    # E. Fusion-training features
    # --------------------------------------------------------

    print(
        "\nPreparing fusion-training inputs...",
        flush=True
    )

    xgb_fusion = xgb_predict(
        xgb, X_fusion_raw
    )

    X_fusion = fusion_features(
        X_fusion_raw,
        xgb_fusion,
        lstm,
        stats
    )

    y_fusion = (
        (
            y_fusion_raw
            - stats["target_mean"]
        ) / stats["target_std"]
    ).astype(np.float32)

    del X_fusion_raw, y_fusion_raw, xgb_fusion
    gc.collect()

    # --------------------------------------------------------
    # F. Validation fusion features
    # --------------------------------------------------------

    print(
        "Preparing validation inputs...",
        flush=True
    )

    xgb_val = xgb_predict(xgb, X_val)

    X_val_fusion = fusion_features(
        X_val,
        xgb_val,
        lstm,
        stats
    )

    y_val_fusion = (
        (
            y_val
            - stats["target_mean"]
        ) / stats["target_std"]
    ).astype(np.float32)

    # --------------------------------------------------------
    # G. Train or load fusion network
    # --------------------------------------------------------

    print("\nTraining/loading fusion...", flush=True)

    if os.path.isfile(fusion_path):

        fusion = HybridFusion().to(DEVICE)

        fusion.load_state_dict(
            torch.load(
                fusion_path,
                map_location=DEVICE,
                weights_only=True
            )
        )

        fusion.eval()
        best_epoch = -1

        print("Saved fusion model loaded.")

    else:

        fusion, best_epoch = train_fusion(
            X_fusion,
            y_fusion,
            X_val_fusion,
            y_val_fusion
        )

        torch.save(
            fusion.state_dict(),
            fusion_path
        )

        print("Fusion model saved.")

    del X_fusion, y_fusion
    del X_val_fusion, y_val_fusion
    gc.collect()

    # --------------------------------------------------------
    # H. VALIDATION EVALUATION
    # --------------------------------------------------------

    print("\nVALIDATION RESULTS")
    print("-" * 55)

    lstm_val, hybrid_val = predict_lstm_hybrid(
        X_val,
        xgb_val,
        lstm,
        fusion,
        stats
    )

    horizon_results = []

    val_models = {
        "Persistence": X_val[
            :, FEATURE_COLS.index("lag_0")
        ],
        "XGBoost": xgb_val,
        "LSTM": lstm_val,
        "Hybrid": hybrid_val
    }

    for model_name, prediction in val_models.items():

        measured = metrics(y_val, prediction)

        row = {
            "Dataset": "PEMS-BAY",
            "Split": "Validation",
            "Horizon": name,
            "Model": model_name,
            "Samples": int(len(y_val)),
            **measured
        }

        horizon_results.append(row)

        print(
            f"{model_name:12s} | "
            f"MAE {measured['MAE']:.4f} | "
            f"RMSE {measured['RMSE']:.4f} | "
            f"MAPE {measured['MAPE (%)']:.4f}%"
        )

    del lstm_val, hybrid_val, xgb_val
    gc.collect()

    # --------------------------------------------------------
    # I. HELD-OUT TEST EVALUATION
    # --------------------------------------------------------

    print("\nTEST RESULTS")
    print("-" * 55)

    xgb_test = xgb_predict(xgb, X_test)

    lstm_test, hybrid_test = predict_lstm_hybrid(
        X_test,
        xgb_test,
        lstm,
        fusion,
        stats
    )

    test_models = {
        "Persistence": X_test[
            :, FEATURE_COLS.index("lag_0")
        ],
        "XGBoost": xgb_test,
        "LSTM": lstm_test,
        "Hybrid": hybrid_test
    }

    for model_name, prediction in test_models.items():

        measured = metrics(
            y_test, prediction
        )

        row = {
            "Dataset": "PEMS-BAY",
            "Split": "Test",
            "Horizon": name,
            "Model": model_name,
            "Samples": int(len(y_test)),
            **measured
        }

        horizon_results.append(row)

        print(
            f"{model_name:12s} | "
            f"MAE {measured['MAE']:.4f} | "
            f"RMSE {measured['RMSE']:.4f} | "
            f"MAPE {measured['MAPE (%)']:.4f}%"
        )

    # --------------------------------------------------------
    # J. SAVE METADATA AND RESULTS
    # --------------------------------------------------------

    metadata = {
        "dataset": "PEMS-BAY",
        "sensors": 325,
        "horizon_minutes": int(horizon * 5),
        "feature_names": FEATURE_COLS,
        "sequence_length": SEQ_LEN,
        "lstm_hidden_size": HIDDEN_SIZE,
        "zero_as_missing": ZERO_AS_MISSING,
        "forward_fill_limit": 6,
        "timezone": "America/Los_Angeles",
        "base_cutoff": str(BASE_CUTOFF),
        "train_cutoff": str(TRAIN_CUTOFF),
        "validation_cutoff": str(VAL_CUTOFF),
        "fusion_start": data["fusion_start"],
        "base_samples": int(len(data["base"][1])),
        "fusion_samples": int(len(data["fusion"][1])),
        "val_samples": int(len(y_val)),
        "test_samples": int(len(y_test)),
        "best_fusion_epoch": int(best_epoch),
        **stats
    }

    save_json(metadata_path, metadata)

    save_json(
        result_path,
        {
            "metadata": metadata,
            "metrics": horizon_results
        }
    )

    all_results.extend(horizon_results)

    pd.DataFrame(all_results).to_csv(
        os.path.join(
            SAVE_DIR,
            "pems_bay_all_results.csv"
        ),
        index=False
    )

    print(
        f"\n{name} models and results saved!"
    )

    # --------------------------------------------------------
    # K. DOWNLOAD CHECKPOINT ZIP AFTER EACH HORIZON
    # --------------------------------------------------------

    checkpoint_zip = (
        f"/content/PEMS_BAY_{name}_Checkpoint.zip"
    )

    with zipfile.ZipFile(
        checkpoint_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as archive:

        for filename in sorted(os.listdir(SAVE_DIR)):

            file_path = os.path.join(
                SAVE_DIR, filename
            )

            if os.path.isfile(file_path):

                archive.write(
                    file_path,
                    arcname=f"pems_bay/{filename}"
                )

    with zipfile.ZipFile(checkpoint_zip) as archive:
        assert archive.testzip() is None

    print(
        "Checkpoint backup:",
        checkpoint_zip
    )

    # Browser may request permission for each download.
    files.download(checkpoint_zip)

    # --------------------------------------------------------
    # L. CLEANUP
    # --------------------------------------------------------

    del data
    del X_val, y_val
    del X_test, y_test

    del xgb, lstm, fusion
    del xgb_test, lstm_test, hybrid_test

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elapsed_minutes = (
        time.time() - start_horizon
    ) / 60

    print(
        f"\n{name} completed in "
        f"{elapsed_minutes:.2f} minutes"
    )


# ============================================================
# 19. FINAL RESULT TABLES
# ============================================================

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 65)
print("PEMS-BAY COMPLETE RESULTS")
print("=" * 65)

display(results_df.round(4))

test_results = results_df[
    results_df["Split"] == "Test"
]

print("\nTEST MAE COMPARISON")

display(
    test_results.pivot(
        index="Horizon",
        columns="Model",
        values="MAE"
    ).round(4)
)

print("\nTEST RMSE COMPARISON")

display(
    test_results.pivot(
        index="Horizon",
        columns="Model",
        values="RMSE"
    ).round(4)
)

print("\nTEST MAPE COMPARISON")

display(
    test_results.pivot(
        index="Horizon",
        columns="Model",
        values="MAPE (%)"
    ).round(4)
)


# ============================================================
# 20. FINAL ZIP BACKUP
# ============================================================

final_csv = os.path.join(
    SAVE_DIR,
    "pems_bay_all_results.csv"
)

results_df.to_csv(
    final_csv,
    index=False
)

final_zip = (
    "/content/PEMS_BAY_Full_Experiment_Backup.zip"
)

shutil.make_archive(
    final_zip.removesuffix(".zip"),
    "zip",
    root_dir="/content/traffic_models",
    base_dir="pems_bay"
)

with zipfile.ZipFile(final_zip) as archive:
    assert archive.testzip() is None

print("\nFINAL BACKUP CREATED")
print("-" * 55)

print("File:", final_zip)
print(
    "Size:",
    round(os.path.getsize(final_zip) / 1024**2, 2),
    "MB"
)

files.download(final_zip)

print("\nALL PEMS-BAY EXPERIMENTS COMPLETED!")


## PEMS-BAY SHAP explainability

Source: original notebook cell 113. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# PEMS-BAY SHAP EXPLAINABILITY ANALYSIS
# XGBoost | 15, 30, 60-minute forecasting
# Same methodology as METR-LA
# No retraining required
# ============================================================

import os
import gc
import json
import zipfile
import random

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# Install SHAP only if needed
try:
    import shap
except ImportError:
    %pip -q install shap
    import shap

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_DIR = "/content/traffic_models/pems_bay"
RESULT_DIR = "/content/PEMS_BAY_SHAP_Results"

os.makedirs(RESULT_DIR, exist_ok=True)

DATA_PATH = "/content/pems-bay-dst-corrected.h5"

if not os.path.exists(DATA_PATH):
    DATA_PATH = "/content/pems-bay.h5"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "PEMS-BAY dataset missing. "
        "Download pems-bay.h5 again."
    )

print("=" * 65)
print("PEMS-BAY SHAP ANALYSIS")
print("=" * 65)

# ============================================================
# 1. LOAD DATASET
# ============================================================

raw = pd.read_hdf(DATA_PATH).astype("float32")

if raw.index.tz is None:
    raw.index = raw.index.tz_localize(
        "America/Los_Angeles",
        ambiguous="raise",
        nonexistent="raise"
    )

assert raw.shape[1] == 325
assert raw.index.is_monotonic_increasing

timestamps = raw.index
n_times = len(raw)

TRAIN_END = int(n_times * 0.70)
VAL_END = int(n_times * 0.80)
BASE_END = int(TRAIN_END * 0.70)

TRAIN_CUTOFF = timestamps[TRAIN_END]
VAL_CUTOFF = timestamps[VAL_END]

raw_values = raw.to_numpy(
    dtype=np.float32,
    copy=False
)

# Match the preprocessing of trained models
valid = np.isfinite(raw_values) & (raw_values != 0)

missing_df = raw.mask(~valid)

base_missing = missing_df.iloc[:BASE_END]

sensor_medians = base_missing.median()

global_median = float(
    base_missing.stack().median()
)

fallback_values = sensor_medians.fillna(
    global_median
)

clean = (
    missing_df
    .ffill(limit=6)
    .fillna(fallback_values)
    .astype(np.float32)
)

clean_values = clean.to_numpy(
    dtype=np.float32,
    copy=False
)

assert np.isfinite(clean_values).all()

print("Dataset:", raw.shape)
print("Validation starts:", TRAIN_CUTOFF)
print("Validation ends:", VAL_CUTOFF)

# ============================================================
# 2. VERIFY FEATURE ORDER
# ============================================================

FEATURE_COLS = [
    f"lag_{i}" for i in range(12)
] + [
    "rolling_mean_3",
    "rolling_std_3",
    "rolling_mean_6",
    "rolling_std_6",
    "rolling_mean_12",
    "rolling_std_12",
    "trend_3",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "weekend"
]

assert len(FEATURE_COLS) == 24

for horizon in ["15min", "30min", "60min"]:

    metadata_path = os.path.join(
        MODEL_DIR,
        f"metadata_{horizon}.json"
    )

    model_path = os.path.join(
        MODEL_DIR,
        f"xgboost_{horizon}.pkl"
    )

    if not os.path.exists(metadata_path):
        raise FileNotFoundError(metadata_path)

    if not os.path.exists(model_path):
        raise FileNotFoundError(model_path)

    with open(metadata_path) as f:
        metadata = json.load(f)

    assert (
        metadata["feature_names"] == FEATURE_COLS
    ), f"Feature order mismatch: {horizon}"

    assert metadata["zero_as_missing"] is True

print("All three model feature orders verified!")

# ============================================================
# 3. CALENDAR FEATURES
# ============================================================

hours = (
    timestamps.hour.to_numpy()
    + timestamps.minute.to_numpy() / 60.0
)

days = timestamps.dayofweek.to_numpy()

calendar = {
    "hour_sin": np.sin(2*np.pi*hours/24),
    "hour_cos": np.cos(2*np.pi*hours/24),
    "day_sin": np.sin(2*np.pi*days/7),
    "day_cos": np.cos(2*np.pi*days/7),
    "weekend": (days >= 5).astype(np.float32)
}

# ============================================================
# 4. CONSTRUCT VALIDATION FEATURES
# ============================================================

HORIZONS = {
    3: "15min",
    6: "30min",
    12: "60min"
}

SAMPLES_PER_SENSOR = 4
EXTRA_SAMPLES = 1035 - 325*3

# Select 3 validation observations per sensor,
# plus one additional observation for 60 sensors.
# Total: 325*3 + 60 = 1035.

rng = np.random.default_rng(SEED)

all_shap_results = []

for horizon, name in HORIZONS.items():

    print("\n" + "=" * 65)
    print("ANALYZING:", name)
    print("=" * 65)

    model = joblib.load(
        os.path.join(
            MODEL_DIR,
            f"xgboost_{name}.pkl"
        )
    )

    samples = []

    target_times = timestamps + pd.Timedelta(
        minutes=5*horizon
    )

    # Same validation target-time definition as training
    time_mask = (
        (target_times >= TRAIN_CUTOFF)
        & (target_times < VAL_CUTOFF)
    )

    for sensor_index in range(325):

        speeds = pd.Series(
            clean_values[:, sensor_index],
            index=timestamps
        )

        features = pd.DataFrame(
            index=timestamps
        )

        for lag in range(12):
            features[f"lag_{lag}"] = speeds.shift(lag)

        for window in [3, 6, 12]:

            rolling = speeds.rolling(window)

            features[f"rolling_mean_{window}"] = (
                rolling.mean()
            )

            features[f"rolling_std_{window}"] = (
                rolling.std()
            )

        features["trend_3"] = (
            speeds - speeds.shift(3)
        )

        for column, values in calendar.items():
            features[column] = values

        valid_target = np.zeros(
            n_times, dtype=bool
        )

        valid_target[:-horizon] = valid[
            horizon:, sensor_index
        ]

        eligible = (
            time_mask
            & valid_target
            & (np.arange(n_times) >= 11)
            & (np.arange(n_times) < n_times-horizon)
        )

        eligible_indices = np.flatnonzero(
            eligible
        )

        # 4 samples from first 60 sensors,
        # 3 from remaining 265 sensors.
        n_samples = 4 if sensor_index < 60 else 3

        if len(eligible_indices) < n_samples:
            raise ValueError(
                f"Insufficient validation samples "
                f"for sensor {sensor_index}"
            )

        chosen = rng.choice(
            eligible_indices,
            size=n_samples,
            replace=False
        )

        selected = features.iloc[
            chosen
        ][FEATURE_COLS].to_numpy(
            dtype=np.float32
        )

        assert np.isfinite(selected).all()

        samples.append(selected)

        del features

    X_sample = np.concatenate(
        samples, axis=0
    )

    assert X_sample.shape == (1035, 24)

    # ========================================================
    # 5. COMPUTE TREE SHAP
    # ========================================================

    print("SHAP sample shape:", X_sample.shape)

    explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(
        X_sample
    )

    shap_values = np.asarray(
        shap_values
    )

    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 0]

    assert shap_values.shape == X_sample.shape

    importance = np.mean(
        np.abs(shap_values),
        axis=0
    )

    result = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "MeanAbsSHAP": importance
    })

    result = result.sort_values(
        "MeanAbsSHAP",
        ascending=False
    ).reset_index(drop=True)

    result.to_csv(
        os.path.join(
            RESULT_DIR,
            f"shap_importance_{name}.csv"
        ),
        index=False
    )

    result_for_comparison = result.copy()
    result_for_comparison["Horizon"] = name

    all_shap_results.append(
        result_for_comparison
    )

    print("\nTOP 10 FEATURES")
    print("-" * 45)

    display(result.head(10).round(4))

    del model, explainer, shap_values
    del X_sample, samples

    gc.collect()


# ============================================================
# 6. COMBINE ALL HORIZONS
# ============================================================

combined = pd.concat(
    all_shap_results,
    ignore_index=True
)

comparison = combined.pivot(
    index="Feature",
    columns="Horizon",
    values="MeanAbsSHAP"
)

comparison = comparison[
    ["15min", "30min", "60min"]
]

comparison["Average"] = comparison.mean(
    axis=1
)

comparison = comparison.sort_values(
    "Average",
    ascending=False
)

comparison.to_csv(
    os.path.join(
        RESULT_DIR,
        "pems_bay_shap_comparison.csv"
    )
)

print("\n" + "=" * 65)
print("TOP FEATURES ACROSS ALL HORIZONS")
print("=" * 65)

display(comparison.head(15).round(4))


# ============================================================
# 7. SHAP COMPARISON FIGURE
# ============================================================

top_features = comparison.head(12)

plot_data = top_features[
    ["15min", "30min", "60min"]
].iloc[::-1]

fig, ax = plt.subplots(
    figsize=(12, 8)
)

y = np.arange(len(plot_data))
bar_height = 0.25

ax.barh(
    y - bar_height,
    plot_data["15min"],
    bar_height,
    label="15 min"
)

ax.barh(
    y,
    plot_data["30min"],
    bar_height,
    label="30 min"
)

ax.barh(
    y + bar_height,
    plot_data["60min"],
    bar_height,
    label="60 min"
)

ax.set_yticks(y)
ax.set_yticklabels(plot_data.index)

ax.set_xlabel(
    "Mean Absolute SHAP Value"
)

ax.set_title(
    "PEMS-BAY: Feature Importance Across Forecast Horizons"
)

ax.legend()
ax.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()

figure_path = os.path.join(
    RESULT_DIR,
    "pems_bay_shap_horizon_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# 8. SAVE EXPERIMENT DETAILS
# ============================================================

shap_metadata = {
    "dataset": "PEMS-BAY",
    "method": "TreeExplainer",
    "explained_model": "XGBoost",
    "does_not_explain": "Full hybrid fusion",
    "sampling_split": "Validation",
    "sample_count_per_horizon": 1035,
    "sensor_count": 325,
    "sampling": (
        "4 observations from first 60 sensors, "
        "3 from each of the other 265 sensors"
    ),
    "random_seed": SEED,
    "feature_order": FEATURE_COLS,
    "timezone": "America/Los_Angeles",
    "zero_as_missing": True,
    "mean_abs_shap_units": (
        "Traffic-speed prediction units"
    )
}

with open(
    os.path.join(RESULT_DIR, "shap_metadata.json"),
    "w"
) as f:
    json.dump(
        shap_metadata,
        f,
        indent=4
    )


# ============================================================
# 9. CREATE SMALL BACKUP ZIP
# ============================================================

zip_path = "/content/PEMS_BAY_SHAP_Results.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for filename in sorted(
        os.listdir(RESULT_DIR)
    ):

        source = os.path.join(
            RESULT_DIR,
            filename
        )

        if os.path.isfile(source):

            archive.write(
                source,
                arcname=filename
            )

with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None

print("\n" + "=" * 65)
print("SHAP ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 65)

print("ZIP file:", zip_path)
print(
    "ZIP size:",
    round(
        os.path.getsize(zip_path)/1024,
        2
    ),
    "KB"
)

print("\nSaved files:")

for name in sorted(os.listdir(RESULT_DIR)):
    print("-", name)

print(
    "\nDownload the ZIP from the Colab Files sidebar."
)


## PEMS-BAY CPU benchmark

Source: original notebook cell 114. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# STAGE 17 — PEMS-BAY MODEL EFFICIENCY BENCHMARK
#
# XGBoost vs LSTM vs Hybrid
# Horizons: 15 / 30 / 60 minutes
#
# CPU inference only — no retraining
# Batch size: 1024
# Warmups: 3 | Repetitions: 10
# ============================================================

import os
import json
import time
import zipfile
import gc

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

torch.set_num_threads(2)

DEVICE = torch.device("cpu")

MODEL_DIR = "/content/traffic_models/pems_bay"
RESULT_DIR = "/content/PEMS_BAY_Efficiency_Results"

os.makedirs(RESULT_DIR, exist_ok=True)

HORIZONS = ["15min", "30min", "60min"]

BATCH_SIZE = 1024
WARMUPS = 3
REPEATS = 10

rng = np.random.default_rng(42)

print("=" * 65)
print("PEMS-BAY MODEL EFFICIENCY BENCHMARK")
print("=" * 65)

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)
print("Warmups:", WARMUPS)
print("Measured repetitions:", REPEATS)
print("Torch CPU threads:", torch.get_num_threads())


# ============================================================
# 1. MODEL ARCHITECTURES
# ============================================================

class TrafficLSTM(nn.Module):

    def __init__(self, hidden_size=64):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :]).squeeze(-1)


class HybridFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(65, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


# ============================================================
# 2. LOAD ALL MODELS AND METADATA
# ============================================================

def load_models(horizon):

    paths = {
        "XGBoost": os.path.join(
            MODEL_DIR, f"xgboost_{horizon}.pkl"
        ),
        "LSTM": os.path.join(
            MODEL_DIR, f"lstm_{horizon}.pth"
        ),
        "Hybrid": os.path.join(
            MODEL_DIR, f"hybrid_{horizon}.pth"
        ),
        "Metadata": os.path.join(
            MODEL_DIR, f"metadata_{horizon}.json"
        )
    }

    for model_name, path in paths.items():
        if not os.path.isfile(path):
            raise FileNotFoundError(
                f"Missing {model_name}: {path}"
            )

    with open(paths["Metadata"]) as file:
        metadata = json.load(file)

    features = metadata["feature_names"]

    assert len(features) == 24
    assert features[:12] == [
        f"lag_{i}" for i in range(12)
    ]

    xgb = joblib.load(paths["XGBoost"])

    lstm = TrafficLSTM(
        metadata["lstm_hidden_size"]
    ).to(DEVICE)

    lstm.load_state_dict(
        torch.load(
            paths["LSTM"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    lstm.eval()

    fusion = HybridFusion().to(DEVICE)

    fusion.load_state_dict(
        torch.load(
            paths["Hybrid"],
            map_location=DEVICE,
            weights_only=True
        )
    )

    fusion.eval()

    return paths, metadata, features, xgb, lstm, fusion


# ============================================================
# 3. CREATE IDENTICAL BENCHMARK INPUTS
# ============================================================

# A fixed synthetic input batch is used for timing.
# It has the correct input shape and plausible speed range.
#
# This measures inference workload, not forecasting accuracy.
# Accuracy was already evaluated on held-out test data.

def make_benchmark_input(feature_names):

    X = np.zeros(
        (BATCH_SIZE, len(feature_names)),
        dtype=np.float32
    )

    # Construct recent traffic speeds with mild variation.
    base_speed = rng.uniform(
        20, 75, size=(BATCH_SIZE, 1)
    ).astype(np.float32)

    lag_values = (
        base_speed
        + rng.normal(
            0, 2,
            size=(BATCH_SIZE, 12)
        ).astype(np.float32)
    )

    lag_values = np.clip(lag_values, 0, 85)

    for i in range(12):
        X[:, feature_names.index(f"lag_{i}")] = (
            lag_values[:, i]
        )

    for window in [3, 6, 12]:

        recent = lag_values[:, :window]

        X[
            :, feature_names.index(f"rolling_mean_{window}")
        ] = recent.mean(axis=1)

        X[
            :, feature_names.index(f"rolling_std_{window}")
        ] = recent.std(axis=1, ddof=1)

    X[:, feature_names.index("trend_3")] = (
        lag_values[:, 0] - lag_values[:, 3]
    )

    hours = rng.uniform(0, 24, BATCH_SIZE)
    days = rng.integers(0, 7, BATCH_SIZE)

    X[:, feature_names.index("hour_sin")] = (
        np.sin(2 * np.pi * hours / 24)
    )

    X[:, feature_names.index("hour_cos")] = (
        np.cos(2 * np.pi * hours / 24)
    )

    X[:, feature_names.index("day_sin")] = (
        np.sin(2 * np.pi * days / 7)
    )

    X[:, feature_names.index("day_cos")] = (
        np.cos(2 * np.pi * days / 7)
    )

    X[:, feature_names.index("weekend")] = (
        (days >= 5).astype(np.float32)
    )

    assert np.isfinite(X).all()

    return X


# ============================================================
# 4. BENCHMARK INFERENCE FUNCTIONS
# ============================================================

def prepare_lstm_input(X, features, metadata):

    lag_indices = [
        features.index(f"lag_{i}")
        for i in range(11, -1, -1)
    ]

    sequences = X[:, lag_indices].copy()

    sequences -= metadata["input_mean"]
    sequences /= metadata["input_std"]

    return torch.from_numpy(
        sequences.reshape(-1, 12, 1)
    )


def xgb_inference(xgb, X):
    return xgb.predict(X)


def lstm_inference(lstm, sequence_tensor):

    with torch.inference_mode():
        return lstm(sequence_tensor)


def hybrid_inference(
    xgb, lstm, fusion,
    X, sequence_tensor, metadata
):

    xgb_predictions = xgb.predict(X)

    with torch.inference_mode():

        lstm_output, _ = lstm.lstm(
            sequence_tensor
        )

        hidden = lstm_output[:, -1, :]

        xgb_normalized = (
            (
                xgb_predictions
                - metadata["target_mean"]
            ) / metadata["target_std"]
        ).astype(np.float32)

        xgb_tensor = torch.from_numpy(
            xgb_normalized
        ).unsqueeze(1)

        combined = torch.cat(
            [hidden, xgb_tensor],
            dim=1
        )

        hybrid_normalized = fusion(combined)

        final_predictions = (
            hybrid_normalized
            * metadata["target_std"]
            + metadata["target_mean"]
        )

        return final_predictions


# ============================================================
# 5. MEASURE MEDIAN LATENCY
# ============================================================

def benchmark(function):

    # Warm up CPU kernels and model execution.
    for _ in range(WARMUPS):
        function()

    durations_ms = []

    for _ in range(REPEATS):

        start = time.perf_counter()

        result = function()

        # Force output materialization when applicable.
        if torch.is_tensor(result):
            result = result.numpy()

        elapsed = (
            time.perf_counter() - start
        ) * 1000

        durations_ms.append(elapsed)

    median_ms = float(
        np.median(durations_ms)
    )

    throughput = (
        BATCH_SIZE / (median_ms / 1000)
    )

    return median_ms, throughput


# ============================================================
# 6. RUN ALL HORIZONS
# ============================================================

records = []

for horizon in HORIZONS:

    print("\n" + "=" * 65)
    print("BENCHMARK:", horizon)
    print("=" * 65)

    paths, metadata, features, xgb, lstm, fusion = (
        load_models(horizon)
    )

    X = make_benchmark_input(features)

    sequence_tensor = prepare_lstm_input(
        X, features, metadata
    )

    operations = {
        "XGBoost": lambda: xgb_inference(xgb, X),

        "LSTM": lambda: lstm_inference(
            lstm, sequence_tensor
        ),

        "Hybrid": lambda: hybrid_inference(
            xgb,
            lstm,
            fusion,
            X,
            sequence_tensor,
            metadata
        )
    }

    model_sizes = {
        "XGBoost": os.path.getsize(
            paths["XGBoost"]
        ) / 1024**2,

        "LSTM": os.path.getsize(
            paths["LSTM"]
        ) / 1024**2,

        "Hybrid": (
            os.path.getsize(paths["XGBoost"])
            + os.path.getsize(paths["LSTM"])
            + os.path.getsize(paths["Hybrid"])
        ) / 1024**2
    }

    for name, operation in operations.items():

        latency, throughput = benchmark(
            operation
        )

        row = {
            "Dataset": "PEMS-BAY",
            "Horizon": horizon,
            "Model": name,
            "Model size (MB)": model_sizes[name],
            "Batch size": BATCH_SIZE,
            "Median latency (ms)": latency,
            "Throughput (predictions/s)": throughput,
            "Warmups": WARMUPS,
            "Repetitions": REPEATS,
            "Device": "CPU"
        }

        records.append(row)

        print(
            f"{name:10s} | "
            f"Size: {model_sizes[name]:.3f} MB | "
            f"Latency: {latency:.3f} ms | "
            f"Throughput: {throughput:.1f}/s"
        )

    del xgb, lstm, fusion
    gc.collect()


# ============================================================
# 7. SAVE CSV RESULTS
# ============================================================

results = pd.DataFrame(records)

csv_path = os.path.join(
    RESULT_DIR,
    "pems_bay_model_efficiency.csv"
)

results.to_csv(csv_path, index=False)

print("\n" + "=" * 65)
print("FINAL EFFICIENCY RESULTS")
print("=" * 65)

display(results.round(3))


# ============================================================
# 8. LATENCY COMPARISON CHART
# ============================================================

latency_table = results.pivot(
    index="Horizon",
    columns="Model",
    values="Median latency (ms)"
)

latency_table = latency_table.loc[
    ["15min", "30min", "60min"],
    ["XGBoost", "LSTM", "Hybrid"]
]

ax = latency_table.plot(
    kind="bar",
    figsize=(11, 6),
    rot=0
)

ax.set_title(
    "PEMS-BAY CPU Inference Latency Comparison"
)

ax.set_xlabel("Forecast Horizon")
ax.set_ylabel("Median inference latency (ms)")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

chart_path = os.path.join(
    RESULT_DIR,
    "pems_bay_latency_comparison.png"
)

plt.savefig(
    chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# 9. SAVE BENCHMARK METADATA
# ============================================================

benchmark_metadata = {
    "dataset": "PEMS-BAY",
    "sensors": 325,
    "horizons": HORIZONS,
    "device": "CPU",
    "batch_size": BATCH_SIZE,
    "warmups": WARMUPS,
    "repetitions": REPEATS,
    "latency_statistic": "median",
    "input_type": "synthetic plausible traffic features",
    "included": (
        "Model inference; hybrid includes XGBoost, "
        "LSTM and fusion calculations"
    ),
    "excluded": (
        "Dataset loading, imputation, feature engineering, "
        "and disk I/O"
    ),
    "model_size_units": "MiB",
    "latency_units": "milliseconds"
}

metadata_path = os.path.join(
    RESULT_DIR,
    "benchmark_metadata.json"
)

with open(metadata_path, "w") as file:
    json.dump(
        benchmark_metadata,
        file,
        indent=4
    )


# ============================================================
# 10. CREATE BACKUP ZIP
# ============================================================

zip_path = "/content/PEMS_BAY_Efficiency_Results.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for filename in sorted(
        os.listdir(RESULT_DIR)
    ):

        file_path = os.path.join(
            RESULT_DIR,
            filename
        )

        archive.write(
            file_path,
            arcname=filename
        )

with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None

print("\nBENCHMARK FINISHED!")
print("CSV:", csv_path)
print("ZIP:", zip_path)

print(
    "ZIP size:",
    round(
        os.path.getsize(zip_path) / 1024,
        2
    ),
    "KB"
)

print(
    "\nDownload the ZIP from the Colab Files sidebar."
)


## PEMS-BAY audit

Source: original notebook cell 115. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# STAGE 18 — FINAL RESEARCH VALIDATION
# PEMS-BAY + METR-LA RESULTS COMPARISON
#
# No retraining
# No Google Drive required
# Verifies model files, splits, feature order,
# zero-speed handling, accuracy, and paper limitations
# ============================================================

import os
import json
import zipfile
import hashlib
import numpy as np
import pandas as pd

MODEL_DIR = "/content/traffic_models/pems_bay"
DATA_PATH = "/content/pems-bay-dst-corrected.h5"
REPORT_DIR = "/content/Research_Validation_Results"

os.makedirs(REPORT_DIR, exist_ok=True)

HORIZONS = ["15min", "30min", "60min"]

EXPECTED_FEATURES = [
    f"lag_{i}" for i in range(12)
] + [
    "rolling_mean_3",
    "rolling_std_3",
    "rolling_mean_6",
    "rolling_std_6",
    "rolling_mean_12",
    "rolling_std_12",
    "trend_3",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "weekend"
]

checks = []

def check(name, condition, details=""):
    passed = bool(condition)
    checks.append({
        "Check": name,
        "Status": "PASS" if passed else "FAIL",
        "Details": str(details)
    })
    print(
        ("PASS" if passed else "FAIL"),
        "-", name,
        details
    )

print("=" * 65)
print("STAGE 18 — FINAL RESEARCH VALIDATION")
print("=" * 65)

# ============================================================
# 1. VERIFY DATASET AND CHRONOLOGICAL SPLITS
# ============================================================

print("\n1. DATASET AND SPLIT VALIDATION")
print("-" * 55)

assert os.path.isfile(DATA_PATH), (
    "DST-corrected PEMS-BAY dataset missing."
)

raw = pd.read_hdf(DATA_PATH)

n = len(raw)

base_end = int(int(n * 0.70) * 0.70)
train_end = int(n * 0.70)
val_end = int(n * 0.80)

base_cutoff = raw.index[base_end]
train_cutoff = raw.index[train_end]
val_cutoff = raw.index[val_end]

check(
    "Expected dataset shape",
    raw.shape == (52116, 325),
    raw.shape
)

check(
    "California timezone",
    str(raw.index.tz) == "America/Los_Angeles",
    raw.index.tz
)

intervals = raw.index.to_series().diff().dropna()

check(
    "Five-minute elapsed intervals",
    (intervals == pd.Timedelta(minutes=5)).all()
)

check(
    "Chronological split ordering",
    base_cutoff < train_cutoff < val_cutoff
)

print("\nBase cutoff:", base_cutoff)
print("Training cutoff:", train_cutoff)
print("Validation cutoff:", val_cutoff)

# ============================================================
# 2. AUDIT ZERO-SPEED OBSERVATIONS
# ============================================================

print("\n2. ZERO-SPEED DATA AUDIT")
print("-" * 55)

partitions = {
    "Base": raw.iloc[:base_end],
    "Fusion-period": raw.iloc[base_end:train_end],
    "Validation": raw.iloc[train_end:val_end],
    "Test": raw.iloc[val_end:]
}

zero_records = []

for name, part in partitions.items():

    arr = part.to_numpy()

    zeros = int(np.count_nonzero(arr == 0))
    nans = int(np.count_nonzero(~np.isfinite(arr)))

    zero_records.append({
        "Partition": name,
        "Observations": int(arr.size),
        "Zero readings": zeros,
        "Zero percentage": 100 * zeros / arr.size,
        "Nonfinite readings": nans
    })

zero_df = pd.DataFrame(zero_records)

display(zero_df.round(6))

check(
    "Total zero readings match original audit",
    int(zero_df["Zero readings"].sum()) == 521
)

# This is a descriptive audit.
# It DOES NOT establish whether zeros are valid speeds.

# ============================================================
# 3. VERIFY SAVED MODELS AND METADATA
# ============================================================

print("\n3. MODEL AND METADATA VALIDATION")
print("-" * 55)

all_metrics = []
metadata_summary = []
model_file_records = []

required_prefixes = {
    "XGBoost": ("xgboost", ".pkl"),
    "LSTM": ("lstm", ".pth"),
    "Hybrid": ("hybrid", ".pth"),
    "Metadata": ("metadata", ".json"),
    "Results": ("results", ".json")
}

for horizon in HORIZONS:

    print("\nHorizon:", horizon)

    paths = {}

    for label, (prefix, ext) in required_prefixes.items():

        path = os.path.join(
            MODEL_DIR,
            f"{prefix}_{horizon}{ext}"
        )

        paths[label] = path

        exists = os.path.isfile(path)

        check(
            f"{horizon} {label} saved",
            exists
        )

        if not exists:
            raise FileNotFoundError(path)

        size = os.path.getsize(path)

        with open(path, "rb") as f:
            digest = hashlib.file_digest(
                f, "sha256"
            ).hexdigest()

        model_file_records.append({
            "Horizon": horizon,
            "File": os.path.basename(path),
            "Size bytes": size,
            "SHA256": digest
        })

    with open(paths["Metadata"]) as f:
        meta = json.load(f)

    with open(paths["Results"]) as f:
        result = json.load(f)

    check(
        f"{horizon} feature order",
        meta["feature_names"] == EXPECTED_FEATURES
    )

    check(
        f"{horizon} base-only preprocessing recorded",
        meta.get("zero_as_missing") is True
        and meta.get("forward_fill_limit") == 6
        and meta.get("base_cutoff") == str(base_cutoff)
    )

    check(
        f"{horizon} split timestamps",
        meta.get("train_cutoff") == str(train_cutoff)
        and meta.get("validation_cutoff") == str(val_cutoff)
    )

    check(
        f"{horizon} model architecture",
        meta.get("sequence_length") == 12
        and meta.get("lstm_hidden_size") == 64
    )

    records = result["metrics"]

    metric_pairs = {
        (r["Split"], r["Model"])
        for r in records
    }

    expected_pairs = {
        (split, model)
        for split in ["Validation", "Test"]
        for model in [
            "Persistence", "XGBoost", "LSTM", "Hybrid"
        ]
    }

    check(
        f"{horizon} complete evaluation",
        metric_pairs == expected_pairs
        and len(records) == 8
    )

    for record in records:

        record = record.copy()
        record["Dataset"] = "PEMS-BAY"

        all_metrics.append(record)

    metadata_summary.append({
        "Horizon": horizon,
        "Base samples": meta["base_samples"],
        "Fusion samples": meta["fusion_samples"],
        "Validation samples": meta["val_samples"],
        "Test samples": meta["test_samples"],
        "Best fusion epoch": meta["best_fusion_epoch"]
    })

metrics_df = pd.DataFrame(all_metrics)

display(pd.DataFrame(metadata_summary))

# ============================================================
# 4. EVALUATE HYBRID AGAINST XGBOOST
# ============================================================

print("\n4. TEST ACCURACY COMPARISON")
print("-" * 55)

test_df = metrics_df[
    metrics_df["Split"] == "Test"
].copy()

display(
    test_df[
        ["Horizon", "Model", "MAE", "RMSE", "MAPE (%)"]
    ].round(4)
)

improvements = []

for horizon in HORIZONS:

    subset = test_df[
        test_df["Horizon"] == horizon
    ].set_index("Model")

    xgb = subset.loc["XGBoost"]
    hybrid = subset.loc["Hybrid"]

    row = {"Horizon": horizon}

    for metric in ["MAE", "RMSE", "MAPE (%)"]:

        row[f"{metric} improvement (%)"] = (
            100 * (xgb[metric] - hybrid[metric])
            / xgb[metric]
        )

    improvements.append(row)

improvement_df = pd.DataFrame(improvements)

print("\nHYBRID IMPROVEMENT OVER XGBOOST")

display(improvement_df.round(4))

check(
    "Hybrid lowers test MAE at all horizons",
    (
        improvement_df["MAE improvement (%)"] > 0
    ).all()
)

check(
    "Hybrid lowers test RMSE at all horizons",
    (
        improvement_df["RMSE improvement (%)"] > 0
    ).all()
)

check(
    "Hybrid lowers test MAPE at all horizons",
    (
        improvement_df["MAPE (%) improvement (%)"] > 0
    ).all()
)

# ============================================================
# 5. CROSS-DATASET SUMMARY
# ============================================================

# These METR-LA results are the previously VERIFIED
# feature-order-corrected held-out test measurements.
# They are included for comparison, not recomputed here.

metr_la_data = [
    ("15min", "XGBoost", 3.1624, 5.7512, 8.5614),
    ("15min", "Hybrid", 3.1761, 5.7410, 8.6033),
    ("30min", "XGBoost", 3.8869, 7.0179, 11.1546),
    ("30min", "Hybrid", 3.8944, 6.9983, 11.0591),
    ("60min", "XGBoost", 4.9104, 8.4909, 14.7218),
    ("60min", "Hybrid", 4.9220, 8.4509, 14.5737)
]

metr_df = pd.DataFrame(
    metr_la_data,
    columns=[
        "Horizon", "Model",
        "MAE", "RMSE", "MAPE (%)"
    ]
)

metr_df["Dataset"] = "METR-LA"

pems_compare = test_df[
    test_df["Model"].isin(
        ["XGBoost", "Hybrid"]
    )
][[
    "Dataset", "Horizon",
    "Model", "MAE", "RMSE", "MAPE (%)"
]]

comparison_df = pd.concat(
    [
        metr_df[
            pems_compare.columns
        ],
        pems_compare
    ],
    ignore_index=True
)

print("\n5. CROSS-DATASET TEST COMPARISON")
print("-" * 55)

display(
    comparison_df.round(4)
)

print(
    "\nNOTE: METR-LA values are previously "
    "measured results, not re-evaluated by this cell."
)

# ============================================================
# 6. VALIDATE MODEL EFFICIENCY REPORT
# ============================================================

print("\n6. EFFICIENCY REPORT CHECK")
print("-" * 55)

efficiency_path = (
    "/content/PEMS_BAY_Efficiency_Results/"
    "pems_bay_model_efficiency.csv"
)

if os.path.isfile(efficiency_path):

    efficiency = pd.read_csv(efficiency_path)

    check(
        "Efficiency benchmark results available",
        len(efficiency) == 9
    )

    display(
        efficiency[
            [
                "Horizon", "Model",
                "Model size (MB)",
                "Median latency (ms)",
                "Throughput (predictions/s)"
            ]
        ].round(3)
    )

else:

    efficiency = None

    print(
        "Efficiency CSV not found in active runtime."
    )
    print(
        "Previous benchmark results remain valid "
        "as reported, but cannot be independently "
        "rechecked in this cell."
    )

# ============================================================
# 7. METHODOLOGY LIMITATIONS
# ============================================================

limitations = [
    {
        "Topic": "Zero-speed assumption",
        "Status": "Sensitivity test pending",
        "Explanation":
            "PEMS-BAY has 521 zero readings. "
            "Current training treats them as missing "
            "and excludes zero targets. Alternative "
            "handling has not been evaluated."
    },
    {
        "Topic": "METR-LA preprocessing",
        "Status": "Correction or disclosure required",
        "Explanation":
            "Original METR-LA fusion experiments "
            "used fallback medians fitted over the "
            "entire training period rather than only "
            "the earlier base-training partition."
    },
    {
        "Topic": "Spatial dependencies",
        "Status": "Scope limitation",
        "Explanation":
            "Models use each sensor's own speed "
            "history and calendar features, without "
            "explicit graph-based neighbor modeling."
    },
    {
        "Topic": "SHAP interpretation",
        "Status": "Scope limitation",
        "Explanation":
            "Tree SHAP explains the XGBoost branch, "
            "not the complete nonlinear fusion model."
    },
    {
        "Topic": "Efficiency",
        "Status": "Measured trade-off",
        "Explanation":
            "Hybrid serialized size is compact, "
            "but CPU batch latency is higher than "
            "standalone XGBoost."
    },
    {
        "Topic": "Statistical uncertainty",
        "Status": "Further validation recommended",
        "Explanation":
            "Paired temporal-block uncertainty "
            "estimates and repeated random seeds "
            "have not been evaluated."
    }
]

limitations_df = pd.DataFrame(limitations)

print("\n7. RESEARCH LIMITATIONS")
print("-" * 55)

display(limitations_df)

# ============================================================
# 8. SAVE AUDIT OUTPUTS
# ============================================================

pd.DataFrame(checks).to_csv(
    f"{REPORT_DIR}/validation_checks.csv",
    index=False
)

zero_df.to_csv(
    f"{REPORT_DIR}/zero_speed_audit.csv",
    index=False
)

metrics_df.to_csv(
    f"{REPORT_DIR}/pems_bay_verified_metrics.csv",
    index=False
)

improvement_df.to_csv(
    f"{REPORT_DIR}/hybrid_improvements.csv",
    index=False
)

comparison_df.to_csv(
    f"{REPORT_DIR}/cross_dataset_comparison.csv",
    index=False
)

limitations_df.to_csv(
    f"{REPORT_DIR}/research_limitations.csv",
    index=False
)

pd.DataFrame(model_file_records).to_csv(
    f"{REPORT_DIR}/model_file_hashes.csv",
    index=False
)

pd.DataFrame(metadata_summary).to_csv(
    f"{REPORT_DIR}/model_metadata_summary.csv",
    index=False
)

# ============================================================
# 9. ZIP REPORT
# ============================================================

zip_path = (
    "/content/Research_Validation_Results.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for name in sorted(os.listdir(REPORT_DIR)):

        path = os.path.join(REPORT_DIR, name)

        if os.path.isfile(path):
            z.write(path, arcname=name)

with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None

failed = sum(
    item["Status"] == "FAIL"
    for item in checks
)

print("\n" + "=" * 65)
print("STAGE 18 VALIDATION COMPLETED")
print("=" * 65)

print("Checks performed:", len(checks))
print("Checks failed:", failed)
print("Report ZIP:", zip_path)
print(
    "ZIP size:",
    round(os.path.getsize(zip_path)/1024, 2),
    "KB"
)

print("\nDownload from the Colab Files sidebar.")

if failed:
    print(
        "\nWARNING: Some validation checks failed. "
        "Review them before using results in the paper."
    )
else:
    print(
        "\nAll implemented consistency checks passed."
    )
    print(
        "Methodological limitations still require "
        "separate research decisions."
    )


## PEMS-BAY zero-target sensitivity

Source: original notebook cell 116. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

import os, json, zipfile, gc
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from IPython.display import display

MODEL_DIR = "/content/traffic_models/pems_bay"
DATA_PATH = "/content/pems-bay-dst-corrected.h5"
OUT_DIR = "/content/PEMS_BAY_Zero_Sensitivity"
os.makedirs(OUT_DIR, exist_ok=True)

torch.set_num_threads(2)
torch.manual_seed(42)

raw = pd.read_hdf(DATA_PATH).astype("float32")
if raw.index.tz is None:
    raw.index = raw.index.tz_localize(
        "America/Los_Angeles"
    )

n = len(raw)
base_end = int(int(n * 0.70) * 0.70)
test_start = int(n * 0.80)

valid = raw.notna() & raw.ne(0)
missing = raw.mask(~valid)
base = missing.iloc[:base_end]

fallback = base.median().fillna(
    float(base.stack().median())
)

clean = (
    missing.ffill(limit=6)
    .fillna(fallback)
    .astype("float32")
)

features = [
    *[f"lag_{i}" for i in range(12)],
    "rolling_mean_3", "rolling_std_3",
    "rolling_mean_6", "rolling_std_6",
    "rolling_mean_12", "rolling_std_12",
    "trend_3", "hour_sin", "hour_cos",
    "day_sin", "day_cos", "weekend"
]

lag_indices = [
    features.index(f"lag_{i}")
    for i in range(11, -1, -1)
]

class TrafficLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            1, 64, batch_first=True
        )
        self.fc = nn.Linear(64, 1)

class HybridFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(65, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

def sensor_features(sensor):
    s = clean.iloc[:, sensor]
    df = pd.DataFrame(index=raw.index)

    for i in range(12):
        df[f"lag_{i}"] = s.shift(i)

    for w in (3, 6, 12):
        roll = s.rolling(w)
        df[f"rolling_mean_{w}"] = roll.mean()
        df[f"rolling_std_{w}"] = roll.std()

    df["trend_3"] = s - s.shift(3)

    idx = raw.index
    hour = idx.hour + idx.minute / 60
    day = idx.dayofweek

    df["hour_sin"] = np.sin(2*np.pi*hour/24)
    df["hour_cos"] = np.cos(2*np.pi*hour/24)
    df["day_sin"] = np.sin(2*np.pi*day/7)
    df["day_cos"] = np.cos(2*np.pi*day/7)
    df["weekend"] = (day >= 5).astype("float32")

    return df[features]

zero_positions = np.argwhere(
    raw.iloc[test_start:].to_numpy() == 0
)

zero_positions[:, 0] += test_start

print("Test zero readings:", len(zero_positions))

records = []
detail_rows = []

for h, name in [(3,"15min"),(6,"30min"),(12,"60min")]:
    print("\nEvaluating", name)

    with open(f"{MODEL_DIR}/metadata_{name}.json") as f:
        meta = json.load(f)

    with open(f"{MODEL_DIR}/results_{name}.json") as f:
        saved = json.load(f)["metrics"]

    assert meta["feature_names"] == features

    # Predict zero targets using observations h steps earlier.
    rows, target_times, sensors = [], [], []
    for sensor in np.unique(zero_positions[:, 1]):
        positions = zero_positions[
            zero_positions[:, 1] == sensor, 0
        ]
        origins = positions - h
        good = origins >= 11

        if not np.any(good):
            continue

        selected = sensor_features(int(sensor)).iloc[
            origins[good]
        ].to_numpy(dtype=np.float32)

        rows.append(selected)
        target_times.extend(raw.index[positions[good]])
        sensors.extend([int(sensor)] * good.sum())

    X = np.concatenate(rows)
    assert np.isfinite(X).all()

    xgb = joblib.load(
        f"{MODEL_DIR}/xgboost_{name}.pkl"
    )
    lstm = TrafficLSTM()
    lstm.load_state_dict(torch.load(
        f"{MODEL_DIR}/lstm_{name}.pth",
        map_location="cpu", weights_only=True
    ))
    lstm.eval()

    fusion = HybridFusion()
    fusion.load_state_dict(torch.load(
        f"{MODEL_DIR}/hybrid_{name}.pth",
        map_location="cpu", weights_only=True
    ))
    fusion.eval()

    xgb_pred = xgb.predict(X).astype("float32")

    seq = X[:, lag_indices].copy()
    seq = (
        (seq - meta["input_mean"])
        / meta["input_std"]
    )

    with torch.inference_mode():
        inp = torch.from_numpy(seq.reshape(-1,12,1))
        output, _ = lstm.lstm(inp)
        hidden = output[:, -1, :]

        lstm_norm = lstm.fc(hidden).squeeze(-1)

        xgb_norm = torch.from_numpy(
            (
                (xgb_pred - meta["target_mean"])
                / meta["target_std"]
            ).reshape(-1,1).astype("float32")
        )

        fusion_norm = fusion(
            torch.cat([hidden,xgb_norm],dim=1)
        )

        lstm_pred = (
            lstm_norm.numpy() * meta["target_std"]
            + meta["target_mean"]
        )
        hybrid_pred = (
            fusion_norm.numpy() * meta["target_std"]
            + meta["target_mean"]
        )

    pred_dict = {
        "Persistence": X[:, 0],
        "XGBoost": xgb_pred,
        "LSTM": lstm_pred,
        "Hybrid": hybrid_pred
    }

    for model_name, predictions in pred_dict.items():

        original = next(
            r for r in saved
            if r["Split"] == "Test"
            and r["Model"] == model_name
        )

        N = int(original["Samples"])
        k = len(predictions)

        # Added targets equal exactly zero.
        added_abs = np.abs(predictions)
        added_sq = np.square(
            predictions.astype(np.float64)
        )

        new_mae = (
            N * original["MAE"]
            + added_abs.sum(dtype=np.float64)
        ) / (N + k)

        new_rmse = np.sqrt(
            (
                N * original["RMSE"]**2
                + added_sq.sum()
            ) / (N + k)
        )

        records.append({
            "Horizon": name,
            "Model": model_name,
            "Original test samples": N,
            "Added zero targets": k,
            "Original MAE": original["MAE"],
            "MAE including zeros": new_mae,
            "Original RMSE": original["RMSE"],
            "RMSE including zeros": new_rmse,
            "MAE change (%)": (
                100*(new_mae-original["MAE"])
                / original["MAE"]
            ),
            "RMSE change (%)": (
                100*(new_rmse-original["RMSE"])
                / original["RMSE"]
            )
        })

        for timestamp, sensor, prediction in zip(
            target_times, sensors, predictions
        ):
            detail_rows.append({
                "Horizon": name,
                "Model": model_name,
                "Target timestamp": str(timestamp),
                "Sensor index": sensor,
                "Actual zero target": 0.0,
                "Prediction": float(prediction)
            })

    del xgb, lstm, fusion
    gc.collect()

result = pd.DataFrame(records)
details = pd.DataFrame(detail_rows)

display(result.round(5))

result.to_csv(
    f"{OUT_DIR}/zero_sensitivity_summary.csv",
    index=False
)

details.to_csv(
    f"{OUT_DIR}/zero_target_predictions.csv",
    index=False
)

notes = {
    "experiment": "PEMS-BAY zero-target evaluation sensitivity",
    "model_retraining": False,
    "zero_targets_assumed_genuine": True,
    "interpretation": (
        "Hypothetical evaluation only. Does not establish "
        "whether zero readings are valid traffic speeds."
    ),
    "mape": (
        "Not recomputed: zero targets have undefined "
        "percentage errors under conventional MAPE."
    ),
    "other_preprocessing": (
        "Uses original trained-model zero-as-missing "
        "input preprocessing."
    ),
    "rounding": (
        "Augmented metrics reconstructed from saved "
        "rounded aggregate metrics."
    )
}

with open(f"{OUT_DIR}/methodology.json","w") as f:
    json.dump(notes, f, indent=2)

zip_path = "/content/PEMS_BAY_Zero_Sensitivity.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for filename in os.listdir(OUT_DIR):
        z.write(
            os.path.join(OUT_DIR, filename),
            arcname=filename
        )

with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None

print("\nSensitivity analysis completed.")
print("Backup:", zip_path)
print("Size:", round(
    os.path.getsize(zip_path)/1024, 2
), "KB")


## PEMS-BAY paired daily uncertainty

Source: original notebook cell 117. Code preserved as authored. Potential dependencies and paths must be reviewed before running.


In [ ]:

# ============================================================
# STAGE 20 — PAIRED DAILY UNCERTAINTY ANALYSIS
# PEMS-BAY | XGBoost vs Hybrid
#
# No retraining, no downloads.
# Samples one hourly prediction origin per sensor.
# Uses day-block bootstrap with 2,000 repetitions.
# ============================================================

import os, json, gc
from collections import defaultdict

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42
BOOTSTRAPS = 2000
HOURLY_STEP = 12
BATCH_SIZE = 8192

torch.set_num_threads(2)

MODEL_DIR = "/content/traffic_models/pems_bay"
OUTPUT_DIR = "/content/traffic_research/uncertainty"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_PATH = "/content/pems-bay-dst-corrected.h5"

if not os.path.isfile(DATA_PATH):
    DATA_PATH = "/content/pems-bay.h5"

assert os.path.isfile(DATA_PATH), (
    "PEMS-BAY dataset not found in this Colab runtime."
)

raw = pd.read_hdf(DATA_PATH).astype("float32")

if raw.index.tz is None:
    raw.index = raw.index.tz_localize(
        "America/Los_Angeles"
    )

assert raw.shape == (52116, 325)

n = len(raw)
base_end = int(int(n * 0.70) * 0.70)
test_start = int(n * 0.80)

raw_values = raw.to_numpy(dtype=np.float32)
valid = np.isfinite(raw_values) & (raw_values != 0)

missing = raw.mask(~valid)
base_missing = missing.iloc[:base_end]

fallback = base_missing.median().fillna(
    float(base_missing.stack().median())
)

clean = (
    missing.ffill(limit=6)
    .fillna(fallback)
    .astype("float32")
)

timestamps = raw.index
clean_values = clean.to_numpy(dtype=np.float32)

FEATURE_COLS = [
    *[f"lag_{i}" for i in range(12)],
    "rolling_mean_3", "rolling_std_3",
    "rolling_mean_6", "rolling_std_6",
    "rolling_mean_12", "rolling_std_12",
    "trend_3", "hour_sin", "hour_cos",
    "day_sin", "day_cos", "weekend"
]

LAG_IDX = [
    FEATURE_COLS.index(f"lag_{i}")
    for i in range(11, -1, -1)
]

hour = (
    timestamps.hour.to_numpy()
    + timestamps.minute.to_numpy() / 60
)
day = timestamps.dayofweek.to_numpy()

calendar = {
    "hour_sin": np.sin(2*np.pi*hour/24),
    "hour_cos": np.cos(2*np.pi*hour/24),
    "day_sin": np.sin(2*np.pi*day/7),
    "day_cos": np.cos(2*np.pi*day/7),
    "weekend": (day >= 5).astype(np.float32)
}

class TrafficLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            1, 64, batch_first=True
        )
        self.fc = nn.Linear(64, 1)

class HybridFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(65, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

def build_features(sensor):
    s = pd.Series(
        clean_values[:, sensor],
        index=timestamps
    )
    df = pd.DataFrame(index=timestamps)

    for i in range(12):
        df[f"lag_{i}"] = s.shift(i)

    for w in (3, 6, 12):
        r = s.rolling(w)
        df[f"rolling_mean_{w}"] = r.mean()
        df[f"rolling_std_{w}"] = r.std()

    df["trend_3"] = s - s.shift(3)

    for key, values in calendar.items():
        df[key] = values

    return df[FEATURE_COLS]

def hybrid_predict(X, xgb, lstm, fusion, meta):
    predictions = np.empty(len(X), dtype=np.float32)

    with torch.inference_mode():
        for start in range(0, len(X), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(X))
            xb = X[start:end]

            xgb_pred = xgb.predict(xb).astype(np.float32)

            seq = xb[:, LAG_IDX].copy()
            seq -= meta["input_mean"]
            seq /= meta["input_std"]

            tensor = torch.from_numpy(
                seq.reshape(-1, 12, 1)
            )

            outputs, _ = lstm.lstm(tensor)
            hidden = outputs[:, -1, :]

            xgb_norm = torch.from_numpy(
                (
                    (xgb_pred - meta["target_mean"])
                    / meta["target_std"]
                ).reshape(-1, 1).astype(np.float32)
            )

            combined = torch.cat(
                [hidden, xgb_norm], dim=1
            )

            pred_norm = fusion(combined)

            predictions[start:end] = (
                pred_norm.numpy() * meta["target_std"]
                + meta["target_mean"]
            )

    return predictions

summary_rows = []
daily_rows = []
rng = np.random.default_rng(SEED)

print("=" * 65)
print("PEMS-BAY PAIRED DAILY UNCERTAINTY ANALYSIS")
print("=" * 65)
print("Hourly sample interval: 12 five-minute steps")
print("Bootstrap repetitions:", BOOTSTRAPS)

for horizon, label in [
    (3, "15min"),
    (6, "30min"),
    (12, "60min")
]:
    print("\n" + "=" * 55)
    print("ANALYZING", label)
    print("=" * 55, flush=True)

    with open(
        f"{MODEL_DIR}/metadata_{label}.json"
    ) as f:
        meta = json.load(f)

    assert meta["feature_names"] == FEATURE_COLS
    assert meta["zero_as_missing"] is True

    xgb = joblib.load(
        f"{MODEL_DIR}/xgboost_{label}.pkl"
    )

    lstm = TrafficLSTM()
    lstm.load_state_dict(torch.load(
        f"{MODEL_DIR}/lstm_{label}.pth",
        map_location="cpu",
        weights_only=True
    ))
    lstm.eval()

    fusion = HybridFusion()
    fusion.load_state_dict(torch.load(
        f"{MODEL_DIR}/hybrid_{label}.pth",
        map_location="cpu",
        weights_only=True
    ))
    fusion.eval()

    # Same target-time split as original experiments.
    # Select one hourly origin on the 5-minute grid.
    candidate_origins = np.arange(
        11, n-horizon
    )

    candidate_origins = candidate_origins[
        (candidate_origins % HOURLY_STEP) == 0
    ]

    target_indices = candidate_origins + horizon

    time_mask = target_indices >= test_start

    candidate_origins = candidate_origins[time_mask]
    target_indices = target_indices[time_mask]

    daily = defaultdict(
        lambda: [0, 0.0, 0.0, 0.0, 0.0]
    )

    for sensor in range(325):
        sensor_valid = valid[
            target_indices, sensor
        ]

        origins = candidate_origins[sensor_valid]
        targets = target_indices[sensor_valid]

        X = build_features(sensor).iloc[
            origins
        ].to_numpy(dtype=np.float32)

        y = raw_values[
            targets, sensor
        ].astype(np.float32)

        assert np.isfinite(X).all()
        assert np.isfinite(y).all()

        xgb_pred = xgb.predict(X)
        hybrid_pred = hybrid_predict(
            X, xgb, lstm, fusion, meta
        )

        error_xgb = np.abs(y - xgb_pred)
        error_hybrid = np.abs(y - hybrid_pred)

        # Group by California local calendar day.
        target_days = timestamps[targets].strftime(
            "%Y-%m-%d"
        )

        for date in np.unique(target_days):
            mask = target_days == date
            entry = daily[date]

            entry[0] += int(mask.sum())
            entry[1] += float(
                error_xgb[mask].sum(dtype=np.float64)
            )
            entry[2] += float(
                error_hybrid[mask].sum(dtype=np.float64)
            )
            entry[3] += float(
                np.square(
                    y[mask] - xgb_pred[mask],
                    dtype=np.float64
                ).sum()
            )
            entry[4] += float(
                np.square(
                    y[mask] - hybrid_pred[mask],
                    dtype=np.float64
                ).sum()
            )

        if (sensor + 1) % 65 == 0:
            print(
                f"Processed {sensor+1}/325 sensors",
                flush=True
            )

    dates = sorted(daily.keys())
    totals = np.array(
        [daily[d] for d in dates],
        dtype=np.float64
    )

    count = totals[:, 0]
    xgb_abs = totals[:, 1]
    hybrid_abs = totals[:, 2]
    xgb_sq = totals[:, 3]
    hybrid_sq = totals[:, 4]

    xgb_mae = xgb_abs.sum() / count.sum()
    hybrid_mae = hybrid_abs.sum() / count.sum()

    improvement = xgb_mae - hybrid_mae

    # Resample whole local days, preserving all the
    # observations within each selected day.
    day_count = len(dates)

    bootstrap_ids = rng.integers(
        0, day_count,
        size=(BOOTSTRAPS, day_count)
    )

    sampled_counts = count[bootstrap_ids].sum(axis=1)

    boot_xgb_mae = (
        xgb_abs[bootstrap_ids].sum(axis=1)
        / sampled_counts
    )

    boot_hybrid_mae = (
        hybrid_abs[bootstrap_ids].sum(axis=1)
        / sampled_counts
    )

    boot_improvement = (
        boot_xgb_mae - boot_hybrid_mae
    )

    ci_low, ci_high = np.percentile(
        boot_improvement, [2.5, 97.5]
    )

    relative_improvement = (
        100 * improvement / xgb_mae
    )

    positive_days = 0

    for i, date in enumerate(dates):
        mae_xgb_day = xgb_abs[i] / count[i]
        mae_hybrid_day = hybrid_abs[i] / count[i]

        positive_days += (
            mae_hybrid_day < mae_xgb_day
        )

        daily_rows.append({
            "Horizon": label,
            "Date": date,
            "Samples": int(count[i]),
            "XGBoost MAE": mae_xgb_day,
            "Hybrid MAE": mae_hybrid_day,
            "MAE improvement": (
                mae_xgb_day - mae_hybrid_day
            ),
            "XGBoost RMSE": np.sqrt(
                xgb_sq[i] / count[i]
            ),
            "Hybrid RMSE": np.sqrt(
                hybrid_sq[i] / count[i]
            )
        })

    summary_rows.append({
        "Horizon": label,
        "Samples": int(count.sum()),
        "Test days": day_count,
        "Days Hybrid better": positive_days,
        "XGBoost sampled MAE": xgb_mae,
        "Hybrid sampled MAE": hybrid_mae,
        "MAE improvement": improvement,
        "Relative improvement (%)": relative_improvement,
        "95% CI lower": ci_low,
        "95% CI upper": ci_high,
        "CI excludes zero": bool(
            ci_low > 0 or ci_high < 0
        )
    })

    print("Samples:", int(count.sum()))
    print("Test days:", day_count)
    print(
        "Days Hybrid better:",
        positive_days, "/", day_count
    )
    print(
        "Sampled XGBoost MAE:",
        round(xgb_mae, 5)
    )
    print(
        "Sampled Hybrid MAE:",
        round(hybrid_mae, 5)
    )
    print(
        "MAE improvement:",
        round(improvement, 5)
    )
    print(
        "95% bootstrap CI:",
        (round(ci_low, 5), round(ci_high, 5))
    )

    del xgb, lstm, fusion
    gc.collect()

summary = pd.DataFrame(summary_rows)
daily_results = pd.DataFrame(daily_rows)

summary.to_csv(
    f"{OUTPUT_DIR}/pems_bay_uncertainty_summary.csv",
    index=False
)

daily_results.to_csv(
    f"{OUTPUT_DIR}/pems_bay_daily_errors.csv",
    index=False
)

# Daily accuracy improvement chart
fig, ax = plt.subplots(figsize=(12, 6))

for label in ["15min", "30min", "60min"]:
    subset = daily_results[
        daily_results["Horizon"] == label
    ]

    ax.plot(
        pd.to_datetime(subset["Date"]),
        subset["MAE improvement"],
        marker=".",
        label=label
    )

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title(
    "PEMS-BAY: Daily MAE Advantage of Hybrid over XGBoost"
)
ax.set_ylabel(
    "XGBoost MAE - Hybrid MAE"
)
ax.set_xlabel("Test date")
ax.legend()
ax.grid(alpha=0.25)

plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/pems_bay_daily_improvement.png",
    dpi=250
)

plt.show()
plt.close()

with open(
    f"{OUTPUT_DIR}/uncertainty_methodology.json",
    "w"
) as f:
    json.dump({
        "dataset": "PEMS-BAY",
        "split": "held-out test",
        "model_retraining": False,
        "sampling": "one hourly origin per sensor",
        "sensors": 325,
        "origin_step": HOURLY_STEP,
        "seed": SEED,
        "bootstrap_unit": "local calendar day",
        "bootstrap_repetitions": BOOTSTRAPS,
        "confidence_interval": "percentile 95%",
        "comparison": (
            "Paired MAE improvement: "
            "XGBoost minus Hybrid"
        ),
        "limitation": (
            "Systematically hourly-sampled test subset, "
            "not full-test inference. "
            "Days may have serial dependence. "
            "The bootstrap is descriptive and "
            "does not account for every dependence."
        )
    }, f, indent=2)

print("\n" + "=" * 65)
print("STAGE 20 COMPLETE")
print("=" * 65)

display(summary.round(5))

print("\nSaved files:", OUTPUT_DIR)

for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)

print(
    "\nNo ZIP downloaded. Results will be included "
    "in the final consolidated research archive."
)
